In [1]:
pip install selenium beautifulsoup4 pandas webdriver-manager


^C
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\samue\Documents\datascientest-lol-draft_analyzer\venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [1]:
import time
import itertools
import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup

import time

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC



In [ ]:
def start_driver():
    options = webdriver.ChromeOptions()

    # IMPORTANT : PAS de headless
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    return driver


driver = start_driver()
driver.get("https://dpm.lol/tierlist?tier=gold_plus")



: 

In [4]:
driver.get("https://dpm.lol/tierlist?tier=gold_plus")


In [5]:
print("⏳ Attente du body...")

body = WebDriverWait(driver, 15).until(
    EC.presence_of_element_located((By.TAG_NAME, "body"))
)

print("✅ Body trouvé")
print(body.text[:500])



⏳ Attente du body...
✅ Body trouvé
Recherchez un Joueur, un Champion, une Équipe, un Pro...
Classé
Arena
ARAM
Tierlist & Builds Or+, 16.2
CHAMPIONS ANALYSÉS
72 132 680
?
Découvrez la tier list de League of Legends de DPM.LOL, mise à jour en direct avec les meilleurs champions pour chaque rôle. Accédez à des builds pro, des runes de haut niveau, des guides d’objets et des ordres de compétences basés sur des données pour améliorer votre rang et dominer vos parties !
GOLD+
TOUT
16.2
Rang
Champion
Voie
Tier
?
Winrate
Pickrate
Parties


In [6]:
from selenium.webdriver.common.by import By
import re

meta_container_selector = (
    "#root > main > div > div.flex.justify-center.gap-16 > "
    "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
    "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
    "div.flex.flex-col.lg\\:flex-row.items-center.justify-between.gap-16.lg\\:gap-24.w-full > "
    "div.flex.flex-row.items-center.justify-center.gap-8.lg\\:gap-16"
)

meta_container = driver.find_element(By.CSS_SELECTOR, meta_container_selector)

# 🔍 on récupère TOUS les spans visibles
spans = meta_container.find_elements(By.TAG_NAME, "span")
span_texts = [s.text.strip() for s in spans if s.text.strip()]

print("🧪 Spans détectés :", span_texts)

# 🎯 identification intelligente
elo = None
server = None
patch = None

for text in span_texts:
    if re.match(r"^\d+\.\d+$", text):          # ex: 14.2
        patch = text
    elif text.upper() in {"EUW", "KR", "NA", "BR", "EUNE", "JP", "LAN", "LAS", "OCE", "RU", "TR", "VN", "TOUT"}:
        server = text
    else:
        elo = text

print("📌 META DETECTÉE")
print(f"   🎯 Elo    : {elo}")
print(f"   🌍 Server : {server}")
print(f"   🧩 Patch  : {patch}")


🧪 Spans détectés : ['GOLD+', 'GOLD+', 'TOUT', 'TOUT', '16.2']
📌 META DETECTÉE
   🎯 Elo    : GOLD+
   🌍 Server : TOUT
   🧩 Patch  : 16.2


In [ ]:
# Ce code commenté marche très bien

from selenium.webdriver.common.by import By
import time

scroll_pause = 1.0
scroll_step = 500

# stockage final
all_rows_data = []

# 🔒 set pour éviter les doublons
seen_champions = set()

container_selector = (
    "#root > main > div > div.flex.justify-center.gap-16 > "
    "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
    "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
    "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
)

last_height = driver.execute_script("return document.body.scrollHeight")
scroll_top = 0

while True:
    driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_top)
    time.sleep(scroll_pause)

    container = driver.find_element(By.CSS_SELECTOR, container_selector)

    # chaque div = 1 champion
    rows = container.find_elements(By.XPATH, "./div/div")
    print(f"📊 Lignes détectées : {len(rows)}")

    for row in rows:
        row_text = row.text.strip()

        # ⛔ déjà vu → on skip
        if row_text in seen_champions:
            continue

        seen_champions.add(row_text)

        svgs = row.find_elements(By.TAG_NAME, "svg")
        svg_html_list = [svg.get_attribute("outerHTML") for svg in svgs]

        all_rows_data.append({
            "text": row_text,
            "svgs": svg_html_list
        })

    scroll_top += scroll_step
    new_height = driver.execute_script("return document.body.scrollHeight")

    if scroll_top >= new_height:
        break

print(f"\n✅ Total champions uniques collectés : {len(all_rows_data)}\n")


📊 Lignes détectées : 19
📊 Lignes détectées : 32
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 34
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 33
📊 Lignes détectées : 30
📊 Lignes détectées : 29
📊 Lignes détectées : 29

✅ Total champions uniques collectés : 208



In [15]:
for i, row in enumerate(all_rows_data):
    print(f"\n🧱 LIGNE {i}")
    print(row["text"])
    print(f"🖼️ Nombre de SVG trouvés : {len(row['svgs'])}")

    for j, svg in enumerate(row["svgs"]):
        print(f"\n--- SVG {j} ---")
        print(svg)


🧱 LIGNE 0
1
Yasuo
17.8%
S+
57.8%
+6.8%
2.0%
422
🖼️ Nombre de SVG trouvés : 2

--- SVG 0 ---
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24" fill="currentColor" class="w-[20px] h-[20px] text-black-300"><path opacity="0.2" d="M17.7929 3C18.2383 3 18.4614 3.53857 18.1464 3.85355L15.1464 6.85355C15.0527 6.94732 14.9255 7 14.7929 7H7.5C7.22386 7 7 7.22386 7 7.5V14.7929C7 14.9255 6.94732 15.0527 6.85355 15.1464L3.85355 18.1464C3.53857 18.4614 3 18.2383 3 17.7929V3.5C3 3.22386 3.22386 3 3.5 3H17.7929Z"></path><path d="M6.20711 21C5.76165 21 5.53857 20.4614 5.85355 20.1464L8.85355 17.1464C8.94732 17.0527 9.0745 17 9.20711 17H16.5C16.7761 17 17 16.7761 17 16.5V9.20711C17 9.0745 17.0527 8.94732 17.1464 8.85355L20.1464 5.85355C20.4614 5.53857 21 5.76165 21 6.20711L21 20.5C21 20.7761 20.7761 21 20.5 21L6.20711 21Z"></path><path opacity="0.2" d="M10 10.5C10 10.2239 10.2239 10 10.5 10H13.5C13.7761 10 14 10.2239 14 10.5V13.5C14 13.7761 13.7761 14 13.5 14H10.5C10.2239 14 10 13.7761 10 13.

In [16]:
ROLE_HASH_TO_TEXT = {
    "54d7bacd7686d25f9555c3381d5b3ecb": "jungle",
    "d1a365179625b6191d515c69f5277dbd": "support",
    "f84094b0fe98e4bf44fe62648e255e41": "adc",
    "6f7f06ca1bef87e71a35726cf843ad87": "top",
    "e4f796e42865301ea9dd362f979a2cdc": "mid",
}

def resolve_role_from_hash(svg_hash: str) -> str:
    role = ROLE_HASH_TO_TEXT.get(svg_hash)

    if role is None:
        print(f"⚠️ Hash de rôle inconnu : {svg_hash}")
        return "unknown"

    return role



In [17]:
import hashlib
from collections import defaultdict

role_svg_map = defaultdict(list)

print(f"🔍 Début traitement de {len(all_rows_data)} champions\n")

for idx, row in enumerate(all_rows_data, start=1):
    print(f"➡️ [{idx}] Champion : {row['text'][:60]}...")

    if len(row["svgs"]) == 0:
        print("   ⛔ Aucun SVG trouvé → skip\n")
        continue

    role_svg = row["svgs"][0]  # SVG DU ROLE
    svg_hash = hashlib.md5(role_svg.encode("utf-8")).hexdigest()
    print(svg_hash)
    print(resolve_role_from_hash(svg_hash))

    # nouveau rôle détecté
    if svg_hash not in role_svg_map:
        print(f"   🆕 Nouveau rôle détecté (hash={svg_hash})")

    role_svg_map[svg_hash].append(row["text"])
    print(f"   ✅ Ajouté au rôle {svg_hash[:8]} "
          f"(total: {len(role_svg_map[svg_hash])})\n")

print("\n📊 RÉSUMÉ FINAL")
print(f"👉 Nombre de rôles distincts : {len(role_svg_map)}")

for i, (svg_hash, champions) in enumerate(role_svg_map.items(), start=1):
    print(f"\n🎭 Rôle #{i} — {resolve_role_from_hash(svg_hash)}")
    print(f"   Champions ({len(champions)}) :")
    for champ in champions:
        print(f"    • {champ}")


🔍 Début traitement de 208 champions

➡️ [1] Champion : 1
Yasuo
17.8%
S+
57.8%
+6.8%
2.0%
422...
f84094b0fe98e4bf44fe62648e255e41
adc
   🆕 Nouveau rôle détecté (hash=f84094b0fe98e4bf44fe62648e255e41)
   ✅ Ajouté au rôle f84094b0 (total: 1)

➡️ [2] Champion : 2
Jayce
24.9%
S+
55.0%
+5.3%
4.4%
920...
6f7f06ca1bef87e71a35726cf843ad87
top
   🆕 Nouveau rôle détecté (hash=6f7f06ca1bef87e71a35726cf843ad87)
   ✅ Ajouté au rôle 6f7f06ca (total: 1)

➡️ [3] Champion : 3
Zoe
76.8%
S+
54.4%
+0.6%
5.7%
1 185...
e4f796e42865301ea9dd362f979a2cdc
mid
   🆕 Nouveau rôle détecté (hash=e4f796e42865301ea9dd362f979a2cdc)
   ✅ Ajouté au rôle e4f796e4 (total: 1)

➡️ [4] Champion : 4
Zyra
51.7%
S+
57.1%
+6.4%
1.3%
268...
54d7bacd7686d25f9555c3381d5b3ecb
jungle
   🆕 Nouveau rôle détecté (hash=54d7bacd7686d25f9555c3381d5b3ecb)
   ✅ Ajouté au rôle 54d7bacd (total: 1)

➡️ [5] Champion : 5
Tristana
79.7%
S+
55.9%
+4.3%
4.8%
989...
f84094b0fe98e4bf44fe62648e255e41
adc
   ✅ Ajouté au rôle f84094b0 (total: 2)

➡️ [6] Ch

In [18]:
def parse_champion_text(text: str) -> dict:
    """
    Attend un texte multi-lignes :
    1: nom du champion
    2: % présence dans le rôle
    3: tier (S+, S, A...)
    4: winrate
    5: pickrate
    6: nombre de parties
    """
    lines = [l.strip() for l in text.split("\n") if l.strip()]

    if len(lines) < 6:
        print("⚠️ Format inattendu :", lines)
        return None

    return {
        "champion": lines[1],
        "role_pickrate": lines[2],
        "tier": lines[3],
        "winrate": lines[4],
        "pickrate": lines[6],
        "games": lines[7],
        "winrate+": lines[5],
    }


In [19]:
import pandas as pd
import hashlib

rows_for_df = []

for row in all_rows_data:
    if len(row["svgs"]) == 0:
        continue

    parsed = parse_champion_text(row["text"])
    if parsed is None:
        continue

    role_svg = row["svgs"][0]
    svg_hash = hashlib.md5(role_svg.encode("utf-8")).hexdigest()
    role = resolve_role_from_hash(svg_hash)

    rows_for_df.append({
        "elo": elo,
        "server": server,
        "patch": patch,
        "champion": parsed["champion"],
        "role": role,
        "role_pickrate": parsed["role_pickrate"],
        "tier": parsed["tier"],
        "winrate": parsed["winrate"],
        "winrate_evol": parsed["winrate+"],
        "pickrate": parsed["pickrate"],
        "games": parsed["games"],
    })

df = pd.DataFrame(rows_for_df)


In [20]:
df.head()

,elo,server,patch,champion,role,role_pickrate,tier,winrate,winrate_evol,pickrate,games
0,CHALLENGER,TOUT,16.2,Yasuo,adc,17.8%,S+,57.8%,+6.8%,2.0%,422
1,CHALLENGER,TOUT,16.2,Jayce,top,24.9%,S+,55.0%,+5.3%,4.4%,920
2,CHALLENGER,TOUT,16.2,Zoe,mid,76.8%,S+,54.4%,+0.6%,5.7%,1 185
3,CHALLENGER,TOUT,16.2,Zyra,jungle,51.7%,S+,57.1%,+6.4%,1.3%,268
4,CHALLENGER,TOUT,16.2,Tristana,adc,79.7%,S+,55.9%,+4.3%,4.8%,989


In [21]:
filename = (
    f"{server}_{elo}_{patch}.csv"
)
df.to_csv(filename, index=False)

---
fin du scrapping, on cherche à itérer
---

In [53]:
from selenium.webdriver.common.by import By
import time

print("🔎 Récupération du container des filtres...")

filters_container_selector = (
    "#root > main > div > div.flex.justify-center.gap-16 > "
    "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
    "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
    "div.flex.flex-col.lg\\:flex-row.items-center.justify-between.gap-16.lg\\:gap-24.w-full > "
    "div.flex.flex-row.items-center.justify-center.gap-8.lg\\:gap-16"
)

filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)
print("✅ Container trouvé")

# =========================
# 1️⃣ CLICK SUR ELO
# =========================

elo_button = filters_container.find_element(By.XPATH, ".//button[1]")
print(f"🟡 Bouton ELO trouvé : '{elo_button.text}'")

print("🖱️ Click ELO")
elo_button.click()

time.sleep(0.5)

# =========================
# 2️⃣ DROPDOWN RADIX
# =========================

radix_id = "radix-_r_0_"
print(f"\n🔽 Recherche du container Radix : #{radix_id}")

radix_root = driver.find_element(By.ID, radix_id)
print("✅ Radix root trouvé")

radix_div = radix_root.find_element(By.XPATH, "./div")
print("✅ Div interne Radix trouvée")

# =========================
# 3️⃣ BOUTONS ELO
# =========================

buttons = radix_div.find_elements(By.XPATH, ".//button")
print(f"\n📊 Nombre de boutons ELO détectés : {len(buttons)}\n")

for i, btn in enumerate(buttons):
    text = btn.text.strip()
    enabled = btn.is_enabled()
    displayed = btn.is_displayed()
    print(f"[{i}] text='{text}' | enabled={enabled} | displayed={displayed}")


🔎 Récupération du container des filtres...
✅ Container trouvé
🟡 Bouton ELO trouvé : 'CHALLENGER'
🖱️ Click ELO

🔽 Recherche du container Radix : #radix-_r_0_
✅ Radix root trouvé
✅ Div interne Radix trouvée

📊 Nombre de boutons ELO détectés : 16

[0] text='Challenger' | enabled=True | displayed=True
[1] text='Grandmaster' | enabled=True | displayed=True
[2] text='Master+' | enabled=True | displayed=True
[3] text='Master' | enabled=True | displayed=True
[4] text='Diamond+' | enabled=True | displayed=True
[5] text='Diamond' | enabled=True | displayed=True
[6] text='Emerald+' | enabled=True | displayed=True
[7] text='Emerald' | enabled=True | displayed=True
[8] text='Platinum+' | enabled=True | displayed=True
[9] text='Platinum' | enabled=True | displayed=True
[10] text='Gold+' | enabled=True | displayed=True
[11] text='Gold' | enabled=True | displayed=True
[12] text='Silver+' | enabled=True | displayed=True
[13] text='Bronze' | enabled=True | displayed=True
[14] text='Iron' | enabled=True 

In [61]:
from selenium.webdriver.common.by import By
import time

print("🔎 Récupération du container des filtres...")

filters_container_selector = (
    "#root > main > div > div.flex.justify-center.gap-16 > "
    "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
    "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
    "div.flex.flex-col.lg\\:flex-row.items-center.justify-between.gap-16.lg\\:gap-24.w-full > "
    "div.flex.flex-row.items-center.justify-center.gap-8.lg\\:gap-16"
)

filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)
print("✅ Container trouvé")

# =========================
# 1️⃣ CLICK SUR ELO
# =========================

elo_button = filters_container.find_element(By.XPATH, ".//button[1]")
print(f"🟡 Bouton ELO trouvé : '{elo_button.text}'")

print("🖱️ Click ELO")
elo_button.click()

time.sleep(0.5)

# =========================
# 2️⃣ DROPDOWN RADIX
# =========================

radix_id = "radix-_r_0_"
print(f"\n🔽 Recherche du container Radix : #{radix_id}")

radix_root = driver.find_element(By.ID, radix_id)
print("✅ Radix root trouvé")

radix_div = radix_root.find_element(By.XPATH, "./div")
print("✅ Div interne Radix trouvée")

# =========================
# 3️⃣ BOUTONS ELO
# =========================

buttons = radix_div.find_elements(By.XPATH, ".//button")
print(f"\n📊 Nombre de boutons ELO détectés : {len(buttons)}\n")

for i, btn in enumerate(buttons):
    text = btn.text.strip()
    enabled = btn.is_enabled()
    displayed = btn.is_displayed()
    print(f"[{i}] text='{text}' | enabled={enabled} | displayed={displayed}")

# =========================
# 4️⃣ CLICK SUR LE PREMIER NOUVEAU BOUTON
# =========================

time.sleep(2)

if buttons:
    first_button = buttons[0]
    print(f"\n🖱️ Click sur le premier bouton Radix : '{first_button.text}'")
    first_button.click()
else:
    print("❌ Aucun bouton Radix trouvé, impossible de cliquer")



🔎 Récupération du container des filtres...
✅ Container trouvé
🟡 Bouton ELO trouvé : 'CHALLENGER'
🖱️ Click ELO

🔽 Recherche du container Radix : #radix-_r_0_
✅ Radix root trouvé
✅ Div interne Radix trouvée

📊 Nombre de boutons ELO détectés : 16

[0] text='Challenger' | enabled=True | displayed=True
[1] text='Grandmaster' | enabled=True | displayed=True
[2] text='Master+' | enabled=True | displayed=True
[3] text='Master' | enabled=True | displayed=True
[4] text='Diamond+' | enabled=True | displayed=True
[5] text='Diamond' | enabled=True | displayed=True
[6] text='Emerald+' | enabled=True | displayed=True
[7] text='Emerald' | enabled=True | displayed=True
[8] text='Platinum+' | enabled=True | displayed=True
[9] text='Platinum' | enabled=True | displayed=True
[10] text='Gold+' | enabled=True | displayed=True
[11] text='Gold' | enabled=True | displayed=True
[12] text='Silver+' | enabled=True | displayed=True
[13] text='Bronze' | enabled=True | displayed=True
[14] text='Iron' | enabled=True 

In [64]:
from selenium.webdriver.common.by import By
import time

print("🔎 Récupération du container des filtres...")

filters_container_selector = (
    "#root > main > div > div.flex.justify-center.gap-16 > "
    "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
    "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
    "div.flex.flex-col.lg\\:flex-row.items-center.justify-between.gap-16.lg\\:gap-24.w-full > "
    "div.flex.flex-row.items-center.justify-center.gap-8.lg\\:gap-16"
)

filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)
print("✅ Container trouvé")

# =========================
# 1️⃣ CLICK SUR SERVER
# =========================

server_button = filters_container.find_element(By.XPATH, ".//button[2]")
print(f"🟡 Bouton SERVER trouvé : '{server_button.text}'")
print("🖱️ Click SERVER")
server_button.click()

time.sleep(0.5)

# =========================
# 2️⃣ DROPDOWN RADIX
# =========================

radix_id = "radix-_r_1_"
print(f"\n🔽 Recherche du container Radix : #{radix_id}")

radix_root = driver.find_element(By.ID, radix_id)
print("✅ Radix root trouvé")

radix_div = radix_root.find_element(By.XPATH, "./div")
print("✅ Div interne Radix trouvée")

# =========================
# 3️⃣ BOUTONS ELO
# =========================

buttons = radix_div.find_elements(By.XPATH, ".//button")
print(f"\n📊 Nombre de boutons ELO détectés : {len(buttons)}\n")

for i, btn in enumerate(buttons):
    text = btn.text.strip()
    enabled = btn.is_enabled()
    displayed = btn.is_displayed()
    print(f"[{i}] text='{text}' | enabled={enabled} | displayed={displayed}")

# =========================
# 4️⃣ CLICK SUR LE PREMIER NOUVEAU BOUTON
# =========================

time.sleep(2)

if buttons:
    first_button = buttons[0]
    print(f"\n🖱️ Click sur le premier bouton Radix : '{first_button.text}'")
    first_button.click()
else:
    print("❌ Aucun bouton Radix trouvé, impossible de cliquer")



🔎 Récupération du container des filtres...
✅ Container trouvé
🟡 Bouton SERVER trouvé : 'EUW'
🖱️ Click SERVER

🔽 Recherche du container Radix : #radix-_r_1_
✅ Radix root trouvé
✅ Div interne Radix trouvée

📊 Nombre de boutons ELO détectés : 13

[0] text='EUW' | enabled=True | displayed=True
[1] text='KR' | enabled=True | displayed=True
[2] text='NA' | enabled=True | displayed=True
[3] text='BR' | enabled=True | displayed=True
[4] text='EUNE' | enabled=True | displayed=True
[5] text='JP' | enabled=True | displayed=True
[6] text='LAN' | enabled=True | displayed=True
[7] text='LAS' | enabled=True | displayed=True
[8] text='OCE' | enabled=True | displayed=True
[9] text='RU' | enabled=True | displayed=True
[10] text='TR' | enabled=True | displayed=True
[11] text='VN' | enabled=True | displayed=True
[12] text='TOUT' | enabled=True | displayed=True

🖱️ Click sur le premier bouton Radix : 'EUW'


In [65]:
# from selenium.webdriver.common.by import By
# import time

# # =========================
# # 1️⃣ CLICK SUR SERVER
# # =========================

# main_buttons = meta_container.find_elements(By.TAG_NAME, "button")
# server_button = main_buttons[1]  # index SERVER
# print(f"🖱️ Click SERVER : '{server_button.text}'")
# server_button.click()

# time.sleep(0.5)

# # =========================
# # 2️⃣ RÉCUPÉRATION DES BOUTONS SERVER (RADIX)
# # =========================

# radix_container = driver.find_element(By.CSS_SELECTOR, "#radix-_r_1_ > div")
# server_buttons = radix_container.find_elements(By.TAG_NAME, "button")

# print(f"\n🌍 Boutons SERVER détectés : {len(server_buttons)}\n")
# for i, btn in enumerate(server_buttons):
#     print(f"[{i}] '{btn.text}'")

# # =========================
# # 3️⃣ CLICK SUR LE PREMIER
# # =========================

# if server_buttons:
#     print(f"\n🖱️ Click premier SERVER : '{server_buttons[0].text}'")
#     server_buttons[0].click()
# else:
#     print("❌ Aucun bouton SERVER trouvé")


In [ ]:
from selenium.webdriver.common.by import By
import time

# =========================
# 1️⃣ CLICK SUR PATCH
# =========================

filters_container_selector = (
    "#root > main > div > div.flex.justify-center.gap-16 > "
    "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
    "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
    "div.flex.flex-col.lg\\:flex-row.items-center.justify-between.gap-16.lg\\:gap-24.w-full > "
    "div.flex.flex-row.items-center.justify-center.gap-8.lg\\:gap-16"
)

filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)
print("✅ Container trouvé")

# =========================
# 1️⃣ CLICK SUR PATCH
# =========================

patch_button = filters_container.find_element(By.XPATH, ".//div/button")
print(f"🟡 Bouton PATCH trouvé : '{patch_button.text}'")
print("🖱️ Click PATCH")
patch_button.click()

time.sleep(0.5)

# =========================
# 2️⃣ RÉCUPÉRATION DES BOUTONS PATCH (RADIX)
# =========================

radix_container = driver.find_element(By.CSS_SELECTOR, "#radix-_r_2_ > div")
patch_buttons = radix_container.find_elements(By.TAG_NAME, "button")

print(f"\n🧩 Boutons PATCH détectés : {len(patch_buttons)}\n")
for i, btn in enumerate(patch_buttons):
    print(f"[{i}] '{btn.text}'")

# =========================
# 3️⃣ CLICK SUR LE PREMIER
# =========================

if patch_buttons:
    print(f"\n🖱️ Click premier PATCH : '{patch_buttons[0].text}'")
    patch_buttons[0].click()
else:
    print("❌ Aucun bouton PATCH trouvé")


✅ Container trouvé
🟡 Bouton PATCH trouvé : '30days'
🖱️ Click PATCH

🧩 Boutons PATCH détectés : 8

[0] '7days'
[1] '14days'
[2] '30days'
[3] '16.2'
[4] '16.1'
[5] '15.24'
[6] '15.23'
[7] '15.22'

🖱️ Click premier PATCH : '7days'


In [68]:
# from selenium.webdriver.common.by import By
# import time

# # =========================
# # 1️⃣ CLICK SUR PATCH
# # =========================

# main_buttons = meta_container.find_elements(By.TAG_NAME, "button")
# patch_button = main_buttons[2]  # index PATCH
# print(f"🖱️ Click PATCH : '{patch_button.text}'")
# patch_button.click()

# time.sleep(0.5)

# # =========================
# # 2️⃣ RÉCUPÉRATION DES BOUTONS PATCH (RADIX)
# # =========================

# radix_container = driver.find_element(By.CSS_SELECTOR, "#radix-_r_2_ > div")
# patch_buttons = radix_container.find_elements(By.TAG_NAME, "button")

# print(f"\n🧩 Boutons PATCH détectés : {len(patch_buttons)}\n")
# for i, btn in enumerate(patch_buttons):
#     print(f"[{i}] '{btn.text}'")

# # =========================
# # 3️⃣ CLICK SUR LE PREMIER
# # =========================

# if patch_buttons:
#     print(f"\n🖱️ Click premier PATCH : '{patch_buttons[0].text}'")
#     patch_buttons[0].click()
# else:
#     print("❌ Aucun bouton PATCH trouvé")


In [ ]:
# def get_last_radix_buttons():
#     root = driver.find_element(By.ID, "root")
#     last_div = root.find_elements(By.XPATH, "./div")[-1]
#     return last_div.find_elements(By.TAG_NAME, "button")




# def get_last_radix_buttons():
#     print("AAA", flush=True)
#     root = driver.find_element(By.ID, "root")
#     all_divs = root.find_elements(By.XPATH, "./div")
#     last_div = all_divs[-1]

#     buttons = last_div.find_elements(By.TAG_NAME, "button")

#     print(f"\n🔽 get_last_radix_buttons() appelé", flush=True)
#     print(f"   Total divs sous #root : {len(all_divs)}", flush=True)
#     print(f"   Div utilisée (dernière) : index {len(all_divs)-1}", flush=True)
#     print(f"   Boutons trouvés dans cette div : {len(buttons)}", flush=True)
#     for i, btn in enumerate(buttons):
#         print(f"     [{i}] '{btn.text.strip()}'", flush=True)

#     return buttons


# def get_last_radix_buttons():
#     """
#     Récupère les boutons du dernier pop-up Radix ouvert.
#     Utilise l'ID commençant par 'radix-' pour identifier le pop-up correct.
#     """
#     print("AAA", flush=True)

#     # 1️⃣ chercher tous les éléments dont l'ID commence par 'radix-'
#     radix_roots = driver.find_elements(By.XPATH, "//*[starts-with(@id, 'radix-')]")
#     if not radix_roots:
#         print("❌ Aucun conteneur Radix trouvé", flush=True)
#         return []

#     # 2️⃣ prendre le dernier pop-up (le plus récemment ouvert)
#     radix_root = radix_roots[-1]

#     # 3️⃣ le div interne qui contient les boutons
#     try:
#         radix_div = radix_root.find_element(By.XPATH, "./div")
#     except:
#         print("❌ Aucun div interne trouvé dans le Radix", flush=True)
#         return []

#     # 4️⃣ récupérer les boutons
#     buttons = radix_div.find_elements(By.TAG_NAME, "button")

#     # 🔽 prints pour debug
#     print(f"\n🔽 get_last_radix_buttons() appelé", flush=True)
#     print(f"   Nombre de Radix trouvés : {len(radix_roots)}", flush=True)
#     print(f"   Radix utilisé : '{radix_root.get_attribute('id')}'", flush=True)
#     print(f"   Nombre de boutons trouvés : {len(buttons)}", flush=True)
#     for i, btn in enumerate(buttons):
#         print(f"     [{i}] '{btn.text.strip()}'", flush=True)

#     return buttons


# def get_last_radix_buttons():
#     """
#     Récupère les boutons du dernier pop-up Radix ouvert.
#     Utilise l'ID commençant par 'radix-' pour identifier le pop-up correct.
#     """

#     # 1️⃣ chercher tous les éléments dont l'ID commence par 'radix-'
#     radix_roots = driver.find_elements(By.XPATH, "//*[starts-with(@id, 'radix-')]")
#     if not radix_roots:

#         return []

#     # 2️⃣ prendre le dernier pop-up (le plus récemment ouvert)
#     radix_root = radix_roots[-1]

#     # 3️⃣ le div interne qui contient les boutons
#     try:
#         radix_div = radix_root.find_element(By.XPATH, "./div")
#     except:

#         return []

#     # 4️⃣ récupérer les boutons
#     buttons = radix_div.find_elements(By.TAG_NAME, "button")

#     return buttons
    




In [20]:
from selenium.webdriver.common.by import By
from collections import defaultdict
import time
import hashlib
import pandas as pd

def scrape_champions(driver, elo, server, patch):
    """
    Scrape les champions affichés pour une combinaison Elo / Server / Patch.
    
    Args:
        driver : Selenium WebDriver actif
        elo : str, nom de l'Elo sélectionné
        server : str, nom du serveur sélectionné
        patch : str, patch sélectionné
    Returns:
        df : pandas.DataFrame avec toutes les données collectées
    """
    scroll_pause = 1.0
    scroll_step = 500

    # stockage final
    all_rows_data = []
    seen_champions = set()

    container_selector = (
        "#root > main > div > div.flex.justify-center.gap-16 > "
        "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
        "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
        "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
    )

    scroll_top = 0
    last_height = driver.execute_script("return document.body.scrollHeight")

    # =========================
    # Scroll et récupération des champions
    # =========================
    while True:
        driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_top)
        time.sleep(scroll_pause)

        container = driver.find_element(By.CSS_SELECTOR, container_selector)
        rows = container.find_elements(By.XPATH, "./div/div")
        print(f"📊 Lignes détectées : {len(rows)}")

        for row in rows:
            row_text = row.text.strip()
            if row_text in seen_champions:
                continue
            seen_champions.add(row_text)

            svgs = row.find_elements(By.TAG_NAME, "svg")
            svg_html_list = [svg.get_attribute("outerHTML") for svg in svgs]

            all_rows_data.append({
                "text": row_text,
                "svgs": svg_html_list
            })

        scroll_top += scroll_step
        new_height = driver.execute_script("return document.body.scrollHeight")
        if scroll_top >= new_height:
            break

    print(f"\n✅ Total champions uniques collectés : {len(all_rows_data)}\n")

    # =========================
    # Mapping rôle
    # =========================
    role_svg_map = defaultdict(list)
    print(f"🔍 Début traitement de {len(all_rows_data)} champions\n")
    for idx, row in enumerate(all_rows_data, start=1):
        if len(row["svgs"]) == 0:
            continue
        role_svg = row["svgs"][0]
        svg_hash = hashlib.md5(role_svg.encode("utf-8")).hexdigest()
        role = resolve_role_from_hash(svg_hash)
        role_svg_map[svg_hash].append(row["text"])

    # =========================
    # Construction du DataFrame
    # =========================
    rows_for_df = []
    for row in all_rows_data:
        if len(row["svgs"]) == 0:
            continue
        parsed = parse_champion_text(row["text"])
        if parsed is None:
            continue
        role_svg = row["svgs"][0]
        svg_hash = hashlib.md5(role_svg.encode("utf-8")).hexdigest()
        role = resolve_role_from_hash(svg_hash)

        rows_for_df.append({
            "elo": elo,
            "server": server,
            "patch": patch,
            "champion": parsed["champion"],
            "role": role,
            "role_pickrate": parsed["role_pickrate"],
            "tier": parsed["tier"],
            "winrate": parsed["winrate"],
            "winrate_evol": parsed["winrate+"],
            "pickrate": parsed["pickrate"],
            "games": parsed["games"],
        })

    df = pd.DataFrame(rows_for_df)

    # =========================
    # Sauvegarde CSV
    # =========================
    filename = f"./winrates/{server}_{elo}_{patch}.csv"
    df.to_csv(filename, index=False)
    print(f"💾 Fichier sauvegardé : {filename}")

    return df


In [ ]:
from selenium.webdriver.common.by import By
from collections import defaultdict
import time
import hashlib
import pandas as pd

def scrape_champions(driver, elo, server, patch):
    """
    Scrape les champions affichés pour une combinaison Elo / Server / Patch.
    
    Args:
        driver : Selenium WebDriver actif
        elo : str, nom de l'Elo sélectionné
        server : str, nom du serveur sélectionné
        patch : str, patch sélectionné
    Returns:
        df : pandas.DataFrame avec toutes les données collectées
    """
    scroll_pause = 1.0
    scroll_step = 500

    # stockage final
    all_rows_data = []
    seen_champions = set()

    container_selector = (
        "#root > main > div > div.flex.justify-center.gap-16 > "
        "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
        "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
        "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
    )

    scroll_top = 0
    last_height = driver.execute_script("return document.body.scrollHeight")

    # =========================
    # Scroll et récupération des champions
    # =========================
    while True:
        driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_top)
        time.sleep(scroll_pause)

        container = driver.find_element(By.CSS_SELECTOR, container_selector)
        rows = container.find_elements(By.XPATH, "./div/div")
        print(f"📊 Lignes détectées : {len(rows)}")

        for row in rows:
            row_text = row.text.strip()
            if row_text in seen_champions:
                continue
            seen_champions.add(row_text)

            svgs = row.find_elements(By.TAG_NAME, "svg")
            svg_html_list = [svg.get_attribute("outerHTML") for svg in svgs]

            all_rows_data.append({
                "text": row_text,
                "svgs": svg_html_list
            })

        scroll_top += scroll_step
        new_height = driver.execute_script("return document.body.scrollHeight")
        if scroll_top >= new_height:
            break

    print(f"\n✅ Total champions uniques collectés : {len(all_rows_data)}\n")

    # =========================
    # Mapping rôle
    # =========================
    role_svg_map = defaultdict(list)
    print(f"🔍 Début traitement de {len(all_rows_data)} champions\n")
    for idx, row in enumerate(all_rows_data, start=1):
        if len(row["svgs"]) == 0:
            continue
        role_svg = row["svgs"][0]
        svg_hash = hashlib.md5(role_svg.encode("utf-8")).hexdigest()
        role = resolve_role_from_hash(svg_hash)
        role_svg_map[svg_hash].append(row["text"])

    # =========================
    # Construction du DataFrame
    # =========================
    rows_for_df = []
    for row in all_rows_data:
        if len(row["svgs"]) == 0:
            continue
        parsed = parse_champion_text(row["text"])
        if parsed is None:
            continue
        role_svg = row["svgs"][0]
        svg_hash = hashlib.md5(role_svg.encode("utf-8")).hexdigest()
        role = resolve_role_from_hash(svg_hash)

        rows_for_df.append({
            "elo": elo,
            "server": server,
            "patch": patch,
            "champion": parsed["champion"],
            "role": role,
            "role_pickrate": parsed["role_pickrate"],
            "tier": parsed["tier"],
            "winrate": parsed["winrate"],
            "winrate_evol": parsed["winrate+"],
            "pickrate": parsed["pickrate"],
            "games": parsed["games"],
        })

    df = pd.DataFrame(rows_for_df)

    # =========================
    # Sauvegarde CSV
    # =========================
    filename = f"./winrates/{server}_{elo}_{patch}.csv"
    df.to_csv(filename, index=False)
    print(f"💾 Fichier sauvegardé : {filename}")

    return df


from typing import Optional, Dict

ROLE_HASH_TO_TEXT = {
    "54d7bacd7686d25f9555c3381d5b3ecb": "jungle",
    "d1a365179625b6191d515c69f5277dbd": "support",
    "f84094b0fe98e4bf44fe62648e255e41": "adc",
    "6f7f06ca1bef87e71a35726cf843ad87": "top",
    "e4f796e42865301ea9dd362f979a2cdc": "mid",
}

def resolve_role_from_hash(svg_hash: str) -> str:
    role = ROLE_HASH_TO_TEXT.get(svg_hash)

    if role is None:
        print(f"⚠️ Hash de rôle inconnu : {svg_hash}")
        return "unknown"

    return role

# def parse_champion_text(text: str) -> dict:
#     """
#     Attend un texte multi-lignes :
#     1: nom du champion
#     2: % présence dans le rôle
#     3: tier (S+, S, A...)
#     4: winrate
#     5: pickrate
#     6: nombre de parties
#     """
#     lines = [l.strip() for l in text.split("\n") if l.strip()]
#     print("LINES :", lines)

#     if len(lines) < 8:
#         print("⚠️ Format inattendu :", lines)
#         print(lines[4])
#         print(lines[4][:-1])  # equivalent to substring(0, len(lines[4])-1)
#         return {
#             "champion": lines[1],
#             "role_pickrate": lines[2],
#             "tier": lines[3],
#             "winrate": lines[4],
#             "pickrate": lines[5],
#             "games": lines[6],
#             "winrate+": '0%',
#         }

#     return {
#         "champion": lines[1],
#         "role_pickrate": lines[2],
#         "tier": lines[3],
#         "winrate": lines[4],
#         "pickrate": lines[6],
#         "games": lines[7],
#         "winrate+": lines[5],
#     }

def parse_champion_text(text: str) -> Optional[Dict]:
    """
    Attend un texte multi-lignes :
    1: nom du champion
    2: % présence dans le rôle
    3: tier (S+, S, A...)
    4: winrate
    5: pickrate
    6: nombre de parties
    """
    try:
        lines = [l.strip() for l in text.split("\n") if l.strip()]
        print("LINES :", lines)

        # ⚠️ format court / inattendu
        if len(lines) < 8:
            print("⚠️ Format inattendu :", lines)

            return {
                "champion": lines[1],
                "role_pickrate": lines[2],
                "tier": lines[3],
                "winrate": lines[4],
                "pickrate": lines[5],
                "games": lines[6],
                "winrate+": "0%",
            }

        # ✅ format normal
        return {
            "champion": lines[1],
            "role_pickrate": lines[2],
            "tier": lines[3],
            "winrate": lines[4],
            "pickrate": lines[6],
            "games": lines[7],
            "winrate+": lines[5],
        }

    except Exception as e:
        print("💥 ERREUR parse_champion_text")
        print("📄 Texte brut :")
        print(text)
        print("❌ Exception :", repr(e))
        return None

def get_last_radix_buttons():
    """
    Récupère les boutons du dernier pop-up Radix ouvert.
    Utilise l'ID commençant par 'radix-' pour identifier le pop-up correct.
    """

    # 1️⃣ chercher tous les éléments dont l'ID commence par 'radix-'
    radix_roots = driver.find_elements(By.XPATH, "//*[starts-with(@id, 'radix-')]")
    if not radix_roots:

        return []

    # 2️⃣ prendre le dernier pop-up (le plus récemment ouvert)
    radix_root = radix_roots[-1]

    # 3️⃣ le div interne qui contient les boutons
    try:
        radix_div = radix_root.find_element(By.XPATH, "./div")
    except:

        return []

    # 4️⃣ récupérer les boutons
    buttons = radix_div.find_elements(By.TAG_NAME, "button")

    return buttons



In [182]:
from selenium.webdriver.common.by import By
import time

import sys
sys.stdout.flush()


# Variables pour suivre la combinaison active
elo = None
server = None
patch = None

# =========================
# Fonction utilitaire pour récupérer les boutons Radix du dernier pop-up
# =========================
# def get_last_radix_buttons():
#     root = driver.find_element(By.ID, "root")
#     last_div = root.find_elements(By.XPATH, "./div")[-1]  # dernier div enfant
#     return last_div.find_elements(By.TAG_NAME, "button")

# =========================
# Récupération des containers
# =========================
filters_container_selector = (
    "#root > main > div > div.flex.justify-center.gap-16 > "
    "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
    "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
    "div.flex.flex-col.lg\\:flex-row.items-center.justify-between.gap-16.lg\\:gap-24.w-full > "
    "div.flex.flex-row.items-center.justify-center.gap-8.lg\\:gap-16"
)
filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)
# meta_container = driver.find_element(By.CSS_SELECTOR, meta_container_selector)

print("✅ Containers trouvés")

# =========================
# 1️⃣ Ouvrir ELO et noter la liste des boutons
# =========================
filters_container.find_element(By.XPATH, ".//button[1]").click()
time.sleep(0.5)
elo_buttons = get_last_radix_buttons()
print(f"\n🎯 ELO détectés : {[b.text for b in elo_buttons]}")

# =========================
# 2️⃣ Ouvrir SERVER et noter la liste des boutons
# =========================
filters_container.find_element(By.XPATH, ".//button[2]").click()
# main_buttons[1].click()
time.sleep(0.5)
server_buttons = get_last_radix_buttons()
print(f"🌍 SERVER détectés : {[b.text for b in server_buttons]}")

# =========================
# 3️⃣ Ouvrir PATCH et noter la liste des boutons
# =========================
filters_container.find_element(By.XPATH, ".//div/button").click()
time.sleep(0.5)
patch_buttons = get_last_radix_buttons()
print(f"🧩 PATCH détectés : {[b.text for b in patch_buttons]}")

# =========================
# BOUCLE SUR TOUTES LES COMBINAISONS
# =========================


# de elo_départ à elo_max
# for i in range(numeloDepart | 0, max(AeloFin, len(elo_buttons))):
# de elo_départ à elo_fin
# for i in range(numeloDepart | 0, min(AelohFin, len(elo_buttons))): 
for i in range(3,max(4, len(elo_buttons))):
    # 🔁 réouvrir la dropdown ELO
    filters_container.find_element(By.XPATH, ".//button[1]").click()
    time.sleep(0.7)
    elo_buttons = get_last_radix_buttons()
    elo_btn = elo_buttons[i]
    elo = elo_btn.text.strip()
    elo_btn.click()
    time.sleep(0.4)
    print(f"\n🎯 ELO [{i}] cliqué → {elo}")

    # de server_départ à server_max
    # for j in range(numServerDepart | 0, max(AServerFin, len(server_buttons))):
    # de server_départ à server_fin
    # for j in range(numServerDepart | 0, min(AServerFin, len(server_buttons))): 
    for j in range(0,max(7, len(server_buttons))):
        if ( ((j >= 3) and (j <= 10) and (j != 4)) ):
            continue  # 🔹 on skip les serveurs non désirés
        # 🔁 réouvrir la dropdown SERVER
        filters_container.find_element(By.XPATH, ".//button[2]").click()
        time.sleep(0.7)
        server_buttons = get_last_radix_buttons()
        server_btn = server_buttons[j]
        server = server_btn.text.strip()
        server_btn.click()
        print(f"  🌍 SERVER [{j}] cliqué → {server}")
        time.sleep(1)
        print("1")
        time.sleep(1)
        print("2")
        time.sleep(1)
        print("3")
        filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)

        # de patch_départ à patch_max
        # for k in range(numPatchDepart | 0, max(APatchFin, len(patch_buttons))):
        # de patch_départ à patch_fin
        # for k in range(numPatchDepart | 0, min(APatchFin, len(patch_buttons))): 
        for k in range(0, max(4, len(patch_buttons))):  # 🔹 on limite à 3 itérations pour tester          
            # 🔁 réouvrir la dropdown PATCH
            filters_container.find_element(By.XPATH, ".//div/button").click()
            time.sleep(1)

            # 🔹 récupérer à nouveau les boutons PATCH pour éviter StaleElementReference
            patch_buttons = get_last_radix_buttons()
            time.sleep(1)
            print("click sur les patchs")
            print(f"🧩 PATCH mis à jour : {[b.text for b in patch_buttons]}")
            patch_btn = patch_buttons[k]

            # 🔹 cliquer sur le kème bouton
            patch = patch_btn.text.strip()
            patch_btn.click()
            time.sleep(0.7)

            print(f"    🧩 PATCH [{k}] cliqué → {patch}")

            # ✅ COMBINAISON ACTIVE
            print(f"    ✅ COMBINAISON ACTIVE : ELO={elo}, SERVER={server}, PATCH={patch}")
            time.sleep(1)
            print("1")
            time.sleep(1)
            print("2")
            time.sleep(1)
            print("3")
            
            filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)
            # Ici tu peux lancer ton scraping pour la combinaison active
            df = scrape_champions(driver, elo, server, patch)
            driver.execute_script("window.scrollTo(0, arguments[0]);", 0)




✅ Containers trouvés

🎯 ELO détectés : ['Challenger', 'Grandmaster', 'Master+', 'Master', 'Diamond+', 'Diamond', 'Emerald+', 'Emerald', 'Platinum+', 'Platinum', 'Gold+', 'Gold', 'Silver+', 'Bronze', 'Iron', 'TOUT']
🌍 SERVER détectés : ['EUW', 'KR', 'NA', 'BR', 'EUNE', 'JP', 'LAN', 'LAS', 'OCE', 'RU', 'TR', 'VN', 'TOUT']


KeyboardInterrupt: 

In [142]:
# ROLE_HASH_TO_TEXT = {
#     "54d7bacd7686d25f9555c3381d5b3ecb": "jungle",
#     "d1a365179625b6191d515c69f5277dbd": "support",
#     "f84094b0fe98e4bf44fe62648e255e41": "adc",
#     "6f7f06ca1bef87e71a35726cf843ad87": "top",
#     "e4f796e42865301ea9dd362f979a2cdc": "mid",
# }

# def resolve_role_from_hash(svg_hash: str) -> str:
#     role = ROLE_HASH_TO_TEXT.get(svg_hash)

#     if role is None:
#         print(f"⚠️ Hash de rôle inconnu : {svg_hash}")
#         return "unknown"

#     return role

# def parse_champion_text(text: str) -> dict:
#     """
#     Attend un texte multi-lignes :
#     1: nom du champion
#     2: % présence dans le rôle
#     3: tier (S+, S, A...)
#     4: winrate
#     5: pickrate
#     6: nombre de parties
#     """
#     lines = [l.strip() for l in text.split("\n") if l.strip()]

#     if len(lines) < 6:
#         print("⚠️ Format inattendu :", lines)
#         return None

#     return {
#         "champion": lines[1],
#         "role_pickrate": lines[2],
#         "tier": lines[3],
#         "winrate": lines[4],
#         "pickrate": lines[6],
#         "games": lines[7],
#         "winrate+": lines[5],
#     }



# # Ce code commenté marche très bien

# from selenium.webdriver.common.by import By
# import time

# scroll_pause = 1.0
# scroll_step = 500

# # stockage final
# all_rows_data = []

# # 🔒 set pour éviter les doublons
# seen_champions = set()

# container_selector = (
#     "#root > main > div > div.flex.justify-center.gap-16 > "
#     "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
#     "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
#     "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
# )

# last_height = driver.execute_script("return document.body.scrollHeight")
# scroll_top = 0

# while True:
#     driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_top)
#     time.sleep(scroll_pause)

#     container = driver.find_element(By.CSS_SELECTOR, container_selector)

#     # chaque div = 1 champion
#     rows = container.find_elements(By.XPATH, "./div/div")
#     print(f"📊 Lignes détectées : {len(rows)}")

#     for row in rows:
#         row_text = row.text.strip()

#         # ⛔ déjà vu → on skip
#         if row_text in seen_champions:
#             continue

#         seen_champions.add(row_text)

#         svgs = row.find_elements(By.TAG_NAME, "svg")
#         svg_html_list = [svg.get_attribute("outerHTML") for svg in svgs]

#         all_rows_data.append({
#             "text": row_text,
#             "svgs": svg_html_list
#         })

#     scroll_top += scroll_step
#     new_height = driver.execute_script("return document.body.scrollHeight")

#     if scroll_top >= new_height:
#         break

# print(f"\n✅ Total champions uniques collectés : {len(all_rows_data)}\n")



# for i, row in enumerate(all_rows_data):
#     print(f"\n🧱 LIGNE {i}")
#     print(row["text"])
#     print(f"🖼️ Nombre de SVG trouvés : {len(row['svgs'])}")

#     for j, svg in enumerate(row["svgs"]):
#         print(f"\n--- SVG {j} ---")
#         print(svg)


# role_svg_map = defaultdict(list)

# print(f"🔍 Début traitement de {len(all_rows_data)} champions\n")

# for idx, row in enumerate(all_rows_data, start=1):
#     print(f"➡️ [{idx}] Champion : {row['text'][:60]}...")

#     if len(row["svgs"]) == 0:
#         print("   ⛔ Aucun SVG trouvé → skip\n")
#         continue

#     role_svg = row["svgs"][0]  # SVG DU ROLE
#     svg_hash = hashlib.md5(role_svg.encode("utf-8")).hexdigest()
#     print(svg_hash)
#     print(resolve_role_from_hash(svg_hash))

#     # nouveau rôle détecté
#     if svg_hash not in role_svg_map:
#         print(f"   🆕 Nouveau rôle détecté (hash={svg_hash})")

#     role_svg_map[svg_hash].append(row["text"])
#     print(f"   ✅ Ajouté au rôle {svg_hash[:8]} "
#           f"(total: {len(role_svg_map[svg_hash])})\n")

# # print("\n📊 RÉSUMÉ FINAL")
# # print(f"👉 Nombre de rôles distincts : {len(role_svg_map)}")

# # for i, (svg_hash, champions) in enumerate(role_svg_map.items(), start=1):
# #     print(f"\n🎭 Rôle #{i} — {resolve_role_from_hash(svg_hash)}")
# #     print(f"   Champions ({len(champions)}) :")
# #     for champ in champions:
# #         print(f"    • {champ}")


# import pandas as pd
# import hashlib

# rows_for_df = []

# for row in all_rows_data:
#     if len(row["svgs"]) == 0:
#         continue

#     parsed = parse_champion_text(row["text"])
#     if parsed is None:
#         continue

#     role_svg = row["svgs"][0]
#     svg_hash = hashlib.md5(role_svg.encode("utf-8")).hexdigest()
#     role = resolve_role_from_hash(svg_hash)

#     rows_for_df.append({
#         "elo": elo,
#         "server": server,
#         "patch": patch,
#         "champion": parsed["champion"],
#         "role": role,
#         "role_pickrate": parsed["role_pickrate"],
#         "tier": parsed["tier"],
#         "winrate": parsed["winrate"],
#         "winrate_evol": parsed["winrate+"],
#         "pickrate": parsed["pickrate"],
#         "games": parsed["games"],
#     })

# df = pd.DataFrame(rows_for_df)


# filename = (
#     f"{server}_{elo}_{patch}.csv"
# )
# df.to_csv(filename, index=False)





In [ ]:
#marche sans le scrapping

# from selenium.webdriver.common.by import By
# import time

# import sys
# sys.stdout.flush()

# # Variables pour suivre la combinaison active
# elo = None
# server = None
# patch = None

# # =========================
# # Fonction utilitaire pour récupérer les boutons Radix du dernier pop-up
# # =========================
# # def get_last_radix_buttons():
# #     root = driver.find_element(By.ID, "root")
# #     last_div = root.find_elements(By.XPATH, "./div")[-1]  # dernier div enfant
# #     return last_div.find_elements(By.TAG_NAME, "button")

# # =========================
# # Récupération des containers
# # =========================
# filters_container_selector = (
#     "#root > main > div > div.flex.justify-center.gap-16 > "
#     "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
#     "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
#     "div.flex.flex-col.lg\\:flex-row.items-center.justify-between.gap-16.lg\\:gap-24.w-full > "
#     "div.flex.flex-row.items-center.justify-center.gap-8.lg\\:gap-16"
# )
# filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)
# # meta_container = driver.find_element(By.CSS_SELECTOR, meta_container_selector)

# print("✅ Containers trouvés")

# # =========================
# # 1️⃣ Ouvrir ELO et noter la liste des boutons
# # =========================
# filters_container.find_element(By.XPATH, ".//button[1]").click()
# time.sleep(0.5)
# elo_buttons = get_last_radix_buttons()
# print(f"\n🎯 ELO détectés : {[b.text for b in elo_buttons]}")

# # =========================
# # 2️⃣ Ouvrir SERVER et noter la liste des boutons
# # =========================
# filters_container.find_element(By.XPATH, ".//button[2]").click()
# # main_buttons[1].click()
# time.sleep(0.5)
# server_buttons = get_last_radix_buttons()
# print(f"🌍 SERVER détectés : {[b.text for b in server_buttons]}")

# # =========================
# # 3️⃣ Ouvrir PATCH et noter la liste des boutons
# # =========================
# filters_container.find_element(By.XPATH, ".//div/button").click()
# time.sleep(0.5)
# patch_buttons = get_last_radix_buttons()
# print(f"🧩 PATCH détectés : {[b.text for b in patch_buttons]}")

# # =========================
# # BOUCLE SUR TOUTES LES COMBINAISONS
# # =========================
# for i, _ in enumerate(elo_buttons):
#     # 🔁 réouvrir la dropdown ELO
#     filters_container.find_element(By.XPATH, ".//button[1]").click()
#     time.sleep(0.7)
#     elo_buttons = get_last_radix_buttons()
#     elo_btn = elo_buttons[i]
#     elo = elo_btn.text.strip()
#     elo_btn.click()
#     time.sleep(0.4)
#     print(f"\n🎯 ELO [{i}] cliqué → {elo}")

#     # for j, _ in enumerate(server_buttons):
#     for j in range(min(3, len(server_buttons))):
#         # 🔁 réouvrir la dropdown SERVER
#         # main_buttons = meta_container.find_elements(By.TAG_NAME, "button")
#         # main_buttons[1].click()
#         filters_container.find_element(By.XPATH, ".//button[2]").click()
#         time.sleep(0.7)
#         server_buttons = get_last_radix_buttons()
#         server_btn = server_buttons[j]
#         server = server_btn.text.strip()
#         server_btn.click()
#         print(f"  🌍 SERVER [{j}] cliqué → {server}")
#         time.sleep(1)
#         print("1")
#         time.sleep(1)
#         print("2")
#         time.sleep(1)
#         print("3")
#         # time.sleep(1)
#         # print("4")
#         # time.sleep(1)
#         # print("5")
#         filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)

#         for k in range(min(3, len(patch_buttons))):  # 🔹 on limite à 3 itérations pour tester
#             # 🔁 réouvrir la dropdown PATCH
#             # main_buttons = meta_container.find_elements(By.TAG_NAME, "button")
#             # main_buttons[2].click()
#             filters_container.find_element(By.XPATH, ".//div/button").click()
#             time.sleep(1)

#             # 🔹 récupérer à nouveau les boutons PATCH pour éviter StaleElementReference
#             patch_buttons = get_last_radix_buttons()
#             time.sleep(1)
#             print("click sur les patchs")
#             print(f"🧩 PATCH mis à jour : {[b.text for b in patch_buttons]}")
#             patch_btn = patch_buttons[k]

#             # 🔹 cliquer sur le kème bouton
#             patch = patch_btn.text.strip()
#             patch_btn.click()
#             time.sleep(0.7)

#             print(f"    🧩 PATCH [{k}] cliqué → {patch}")

#             # ✅ COMBINAISON ACTIVE
#             print(f"    ✅ COMBINAISON ACTIVE : ELO={elo}, SERVER={server}, PATCH={patch}")
#             time.sleep(1)
#             print("1")
#             time.sleep(1)
#             print("2")
#             time.sleep(1)
#             print("3")
#             filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)


#             # Ici tu peux lancer ton scraping pour la combinaison active


#         # for k, _ in enumerate(patch_buttons):
#         #     # 🔁 réouvrir la dropdown PATCH
#         #     main_buttons = meta_container.find_elements(By.TAG_NAME, "button")
#         #     main_buttons[2].click()
#         #     time.sleep(0.7)
#         #     patch_buttons = get_last_radix_buttons()
#         #     patch_btn = patch_buttons[k]
#         #     patch = patch_btn.text.strip()
#         #     patch_btn.click()
#         #     time.sleep(0.4)
#         #     print(f"    🧩 PATCH [{k}] cliqué → {patch}")

#         #     # ✅ COMBINAISON ACTIVE
#         #     print(f"    ✅ COMBINAISON ACTIVE : ELO={elo}, SERVER={server}, PATCH={patch}")

#         #     # Ici tu peux lancer ton scraping pour la combinaison active


---
Maintenant, la collect des matchups et synergies
---

In [ ]:
# def start_driver():
#     options = webdriver.ChromeOptions()

#     # IMPORTANT : PAS de headless
#     options.add_argument("--start-maximized")
#     options.add_argument("--disable-blink-features=AutomationControlled")

#     driver = webdriver.Chrome(
#         service=Service(ChromeDriverManager().install()),
#         options=options
#     )
#     return driver


# driver = start_driver()
# driver.get("https://dpm.lol/tierlist?tier=gold_plus")

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import WebDriverException
from webdriver_manager.chrome import ChromeDriverManager
import concurrent.futures
import time

DEBUG_PORT = 9222

def try_connect_existing_chrome():
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option(
        "debuggerAddress", f"127.0.0.1:{DEBUG_PORT}"
    )
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


def get_or_create_driver(timeout=5):
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            print("🔁 Tentative de connexion à Chrome existant...")
            with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
                future = executor.submit(try_connect_existing_chrome)
                driver = future.result(timeout=timeout)
            print("✅ Connecté à Chrome existant")
            return driver
        except (WebDriverException, concurrent.futures.TimeoutError):
            print("⏳ Chrome non dispo ou timeout, retry...")
            time.sleep(0.5)

    # Après timeout → lancement d'un nouveau Chrome
    print("🚀 Timeout atteint → lancement d'un nouveau Chrome")
    options = webdriver.ChromeOptions()
    options.add_argument(f"--remote-debugging-port={DEBUG_PORT}")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    print("🆕 Nouveau Chrome lancé avec debugging")
    return driver


In [3]:
driver = get_or_create_driver()
driver.get("https://dpm.lol/tierlist?tier=gold_plus")

🔁 Tentative de connexion à Chrome existant...
⏳ Chrome non dispo ou timeout, retry...
🚀 Timeout atteint → lancement d'un nouveau Chrome
🆕 Nouveau Chrome lancé avec debugging


In [4]:
from selenium.webdriver.common.by import By
from collections import defaultdict
import time
import hashlib
import pandas as pd

def scrape_champions(driver, elo, server, patch):
    """
    Scrape les champions affichés pour une combinaison Elo / Server / Patch.
    
    Args:
        driver : Selenium WebDriver actif
        elo : str, nom de l'Elo sélectionné
        server : str, nom du serveur sélectionné
        patch : str, patch sélectionné
    Returns:
        df : pandas.DataFrame avec toutes les données collectées
    """
    scroll_pause = 1.0
    scroll_step = 500

    # stockage final
    all_rows_data = []
    seen_champions = set()

    container_selector = (
        "#root > main > div > div.flex.justify-center.gap-16 > "
        "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
        "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
        "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
    )

    scroll_top = 0
    last_height = driver.execute_script("return document.body.scrollHeight")

    # =========================
    # Scroll et récupération des champions
    # =========================
    while True:
        driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_top)
        time.sleep(scroll_pause)

        container = driver.find_element(By.CSS_SELECTOR, container_selector)
        rows = container.find_elements(By.XPATH, "./div/div")
        print(f"📊 Lignes détectées : {len(rows)}")

        for row in rows:
            row_text = row.text.strip()
            if row_text in seen_champions:
                continue
            seen_champions.add(row_text)

            svgs = row.find_elements(By.TAG_NAME, "svg")
            svg_html_list = [svg.get_attribute("outerHTML") for svg in svgs]

            all_rows_data.append({
                "text": row_text,
                "svgs": svg_html_list
            })

        scroll_top += scroll_step
        new_height = driver.execute_script("return document.body.scrollHeight")
        if scroll_top >= new_height:
            break

    print(f"\n✅ Total champions uniques collectés : {len(all_rows_data)}\n")

    # =========================
    # Mapping rôle
    # =========================
    role_svg_map = defaultdict(list)
    print(f"🔍 Début traitement de {len(all_rows_data)} champions\n")
    for idx, row in enumerate(all_rows_data, start=1):
        if len(row["svgs"]) == 0:
            continue
        role_svg = row["svgs"][0]
        svg_hash = hashlib.md5(role_svg.encode("utf-8")).hexdigest()
        role = resolve_role_from_hash(svg_hash)
        role_svg_map[svg_hash].append(row["text"])

    # =========================
    # Construction du DataFrame
    # =========================
    rows_for_df = []
    for row in all_rows_data:
        if len(row["svgs"]) == 0:
            continue
        parsed = parse_champion_text(row["text"])
        if parsed is None:
            continue
        role_svg = row["svgs"][0]
        svg_hash = hashlib.md5(role_svg.encode("utf-8")).hexdigest()
        role = resolve_role_from_hash(svg_hash)

        rows_for_df.append({
            "elo": elo,
            "server": server,
            "patch": patch,
            "champion": parsed["champion"],
            "role": role,
            "role_pickrate": parsed["role_pickrate"],
            "tier": parsed["tier"],
            "winrate": parsed["winrate"],
            "winrate_evol": parsed["winrate+"],
            "pickrate": parsed["pickrate"],
            "games": parsed["games"],
        })

    df = pd.DataFrame(rows_for_df)

    # =========================
    # Sauvegarde CSV
    # =========================
    filename = f"./winrates/{server}_{elo}_{patch}.csv"
    df.to_csv(filename, index=False)
    print(f"💾 Fichier sauvegardé : {filename}")

    return df


from typing import Optional, Dict

ROLE_HASH_TO_TEXT = {
    "54d7bacd7686d25f9555c3381d5b3ecb": "jungle",
    "d1a365179625b6191d515c69f5277dbd": "support",
    "f84094b0fe98e4bf44fe62648e255e41": "adc",
    "6f7f06ca1bef87e71a35726cf843ad87": "top",
    "e4f796e42865301ea9dd362f979a2cdc": "mid",
}

def resolve_role_from_hash(svg_hash: str) -> str:
    role = ROLE_HASH_TO_TEXT.get(svg_hash)

    if role is None:
        print(f"⚠️ Hash de rôle inconnu : {svg_hash}")
        return "unknown"

    return role


def parse_champion_text(text: str) -> Optional[Dict]:
    """
    Attend un texte multi-lignes :
    1: nom du champion
    2: % présence dans le rôle
    3: tier (S+, S, A...)
    4: winrate
    5: pickrate
    6: nombre de parties
    """
    try:
        lines = [l.strip() for l in text.split("\n") if l.strip()]
        print("LINES :", lines)

        # ⚠️ format court / inattendu
        if len(lines) < 8:
            print("⚠️ Format inattendu :", lines)

            return {
                "champion": lines[1],
                "role_pickrate": lines[2],
                "tier": lines[3],
                "winrate": lines[4],
                "pickrate": lines[5],
                "games": lines[6],
                "winrate+": "0%",
            }

        # ✅ format normal
        return {
            "champion": lines[1],
            "role_pickrate": lines[2],
            "tier": lines[3],
            "winrate": lines[4],
            "pickrate": lines[6],
            "games": lines[7],
            "winrate+": lines[5],
        }

    except Exception as e:
        print("💥 ERREUR parse_champion_text")
        print("📄 Texte brut :")
        print(text)
        print("❌ Exception :", repr(e))
        return None

def get_last_radix_buttons():
    """
    Récupère les boutons du dernier pop-up Radix ouvert.
    Utilise l'ID commençant par 'radix-' pour identifier le pop-up correct.
    """

    # 1️⃣ chercher tous les éléments dont l'ID commence par 'radix-'
    radix_roots = driver.find_elements(By.XPATH, "//*[starts-with(@id, 'radix-')]")
    if not radix_roots:

        return []

    # 2️⃣ prendre le dernier pop-up (le plus récemment ouvert)
    radix_root = radix_roots[-1]

    # 3️⃣ le div interne qui contient les boutons
    try:
        radix_div = radix_root.find_element(By.XPATH, "./div")
    except:

        return []

    # 4️⃣ récupérer les boutons
    buttons = radix_div.find_elements(By.TAG_NAME, "button")

    return buttons



In [ ]:
# from selenium.webdriver.common.by import By
# import time

# def click_liste_complete(driver):
#     print("🟢 Recherche du bouton '+ Liste complète' parmi tous les boutons...", flush=True)

#     buttons = driver.find_elements(By.TAG_NAME, "button")
#     print(f"🔍 {len(buttons)} boutons trouvés sur la page")

#     target_button = None
#     for i, btn in enumerate(buttons):
#         btn_text = btn.text.strip()
#         # normalisation pour enlever retours à la ligne et espaces multiples
#         btn_text_clean = " ".join(btn_text.split())
#         print(f"[{i}] → '{btn_text_clean}'")
#         if "+ Liste complète" in btn_text_clean:
#             target_button = btn
#             break

#     if target_button:
#         driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", target_button)
#         time.sleep(0.75)
#         target_button.click()
#         print("🖱️ Bouton '+ Liste complète' cliqué", flush=True)
#         time.sleep(1)
#     else:
#         print("❌ Aucun bouton '+ Liste complète' trouvé", flush=True)








# from selenium.webdriver.common.by import By
# from selenium.common.exceptions import (
#     NoSuchElementException,
#     StaleElementReferenceException,
#     ElementClickInterceptedException,
#     ElementNotInteractableException,
# )
# import time


# def click_liste_complete(driver):
#     print("🟢 Recherche du bouton '+ Liste complète' parmi tous les boutons...", flush=True)

#     for attempt in range(2):
#         print(f"🔁 Tentative {attempt + 1}/2")

#         try:
#             buttons = driver.find_elements(By.TAG_NAME, "button")
#             print(f"🔍 {len(buttons)} boutons trouvés sur la page")

#             for i, btn in enumerate(buttons):
#                 try:
#                     btn_text = btn.text.strip()
#                     btn_text_clean = " ".join(btn_text.split())

#                     # XPath absolu de debug
#                     btn_xpath = driver.execute_script(
#                         """
#                         function getXPath(element) {
#                             if (element.id !== '')
#                                 return '//*[@id="' + element.id + '"]';
#                             if (element === document.body)
#                                 return '/html/body';

#                             let ix = 0;
#                             let siblings = element.parentNode.childNodes;
#                             for (let i = 0; i < siblings.length; i++) {
#                                 let sibling = siblings[i];
#                                 if (sibling === element)
#                                     return getXPath(element.parentNode) + '/' +
#                                         element.tagName.toLowerCase() + '[' + (ix + 1) + ']';
#                                 if (sibling.nodeType === 1 && sibling.tagName === element.tagName)
#                                     ix++;
#                             }
#                         }
#                         return getXPath(arguments[0]);
#                         """,
#                         btn,
#                     )

#                     print(f"[{i}] → '{btn_text_clean}' | XPath: {btn_xpath}")

#                     if "+ Liste complète" in btn_text_clean:
#                         print("🎯 Bouton cible détecté, tentative de clic...")

#                         driver.execute_script(
#                             "arguments[0].scrollIntoView({block: 'center'});",
#                             btn,
#                         )
#                         time.sleep(0.75)
#                         btn.click()

#                         print("🖱️ Bouton '+ Liste complète' cliqué avec succès", flush=True)
#                         time.sleep(1)
#                         return True

#                 except StaleElementReferenceException:
#                     print(f"⚠️ Bouton [{i}] devenu obsolète (DOM mis à jour)")
#                 except Exception as e:
#                     print(f"⚠️ Erreur sur le bouton [{i}] : {e}")

#         except (
#             NoSuchElementException,
#             ElementClickInterceptedException,
#             ElementNotInteractableException,
#         ) as e:
#             print(f"❌ Erreur lors de la tentative {attempt + 1} : {e}")

#         time.sleep(0.5)

#     print("❌ Bouton '+ Liste complète' introuvable ou non cliquable après 2 tentatives", flush=True)
#     return False



In [5]:
from selenium.webdriver.common.by import By
from selenium.common.exceptions import (
    NoSuchElementException,
    StaleElementReferenceException,
    ElementClickInterceptedException,
    ElementNotInteractableException,
)
import time


def click_liste_complete(driver) -> bool:
    print("🟢 Recherche du bouton '+ Liste complète' parmi tous les boutons...", flush=True)

    for attempt in range(2):
        print(f"🔁 Tentative {attempt + 1}/2")

        try:
            buttons = driver.find_elements(By.TAG_NAME, "button")
            print(f"🔍 {len(buttons)} boutons trouvés sur la page")

            for i, btn in enumerate(buttons):
                try:
                    btn_text = btn.text.strip()
                    btn_text_clean = " ".join(btn_text.split())

                    # XPath absolu (debug)
                    btn_xpath = driver.execute_script(
                        """
                        function getXPath(element) {
                            if (element.id !== '')
                                return '//*[@id="' + element.id + '"]';
                            if (element === document.body)
                                return '/html/body';

                            let ix = 0;
                            let siblings = element.parentNode.childNodes;
                            for (let i = 0; i < siblings.length; i++) {
                                let sibling = siblings[i];
                                if (sibling === element)
                                    return getXPath(element.parentNode) + '/' +
                                        element.tagName.toLowerCase() + '[' + (ix + 1) + ']';
                                if (sibling.nodeType === 1 && sibling.tagName === element.tagName)
                                    ix++;
                            }
                        }
                        return getXPath(arguments[0]);
                        """,
                        btn,
                    )

                    print(f"[{i}] → '{btn_text_clean}' | XPath: {btn_xpath}")

                    if "+ Liste complète" in btn_text_clean:
                        print("🎯 Bouton '+ Liste complète' détecté, tentative de clic...")

                        driver.execute_script(
                            "arguments[0].scrollIntoView({block: 'center'});",
                            btn,
                        )
                        time.sleep(0.75)
                        btn.click()

                        print("🖱️ Bouton '+ Liste complète' cliqué avec succès", flush=True)
                        time.sleep(1)
                        return True  # ✅ succès immédiat

                except StaleElementReferenceException:
                    print(f"⚠️ Bouton [{i}] devenu obsolète (DOM mis à jour)")
                except Exception as e:
                    print(f"⚠️ Erreur sur le bouton [{i}] : {e}")

        except (
            NoSuchElementException,
            ElementClickInterceptedException,
            ElementNotInteractableException,
        ) as e:
            print(f"❌ Erreur lors de la tentative {attempt + 1} : {e}")

        time.sleep(0.5)

    print("❌ Bouton '+ Liste complète' non cliqué après 2 tentatives", flush=True)
    return False  # ❌ échec final


In [ ]:
# from selenium.webdriver.common.by import By
# from selenium.webdriver.common.action_chains import ActionChains
# from selenium.webdriver.common.keys import Keys
# from selenium.common.exceptions import StaleElementReferenceException
# import time


# def open_champion_in_new_tab(
#     driver,
#     champion_descriptor: dict,
#     wait_seconds: int = 5,
# ):
#     """
#     champion_descriptor = {
#         "global_index": int,
#         "scroll_y": int,
#         "label": str
#     }
#     """

#     container_selector = (
#         "#root > main > div > div.flex.justify-center.gap-16 > "
#         "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
#         "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
#         "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
#     )

#     scroll_y = champion_descriptor["scroll_y"]
#     label = champion_descriptor["label"]
#     numero= champion_descriptor["numero"]

#     print(f"🧭 Scroll vers Y={scroll_y} pour {label}")
#     driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_y)
#     time.sleep(0.5)  # 👈 important

#     container = driver.find_element(By.CSS_SELECTOR, container_selector)
#     rows = container.find_elements(By.XPATH, "./div/div")

#     if not rows:
#         print("❌ Aucun champion visible après scroll")
#         return

#     row = rows[numero]  # premier visible à cet endroit
#     print(f"🖱️ Ouverture champion → {label}")

#     main_window = driver.current_window_handle

#     ActionChains(driver) \
#         .key_down(Keys.CONTROL) \
#         .click(row) \
#         .key_up(Keys.CONTROL) \
#         .perform()

#     time.sleep(2)

#     windows = driver.window_handles
#     if len(windows) < 2:
#         print("❌ Nouvel onglet non détecté")
#         return

#     new_tab = [w for w in windows if w != main_window][0]

#     for i in range(1, wait_seconds + 1):
#         print(f"⏳ {i}/{wait_seconds}s")
#         time.sleep(1)
#     driver.switch_to.window(new_tab)
#     for i in range(1, wait_seconds + 1):
#         print(f"⏳ {i}/{wait_seconds}s")
#         time.sleep(1)
#     # print_all_clickable_elements(driver, timeout=5)
#     # deploiement_counter_synergies(driver)
#     click_liste_complete(driver)

#     time.sleep(1)
#     matchups = collect_matchup(driver)
#     print(f"📦 Matchups récupérés : {len(matchups)}")

#     for i in range(1, wait_seconds + 3):
#         print(f"⏳ {i}/{wait_seconds}s")
#         time.sleep(1)
#     driver.close()
#     time.sleep(1)
#     driver.switch_to.window(main_window)
#     print("↩️ Retour liste champions")


In [ ]:
# sauvegarde avant de tester la sauvegarde des données

# from selenium.webdriver.common.by import By
# from selenium.webdriver.common.action_chains import ActionChains
# from selenium.webdriver.common.keys import Keys
# from selenium.common.exceptions import StaleElementReferenceException
# import time


# def open_champion_in_new_tab(
#     driver,
#     champion_descriptor: dict,
#     wait_seconds: int = 5,
# ):
#     """
#     champion_descriptor = {
#         "global_index": int,
#         "scroll_y": int,
#         "label": str
#     }
#     """

#     container_selector = (
#         "#root > main > div > div.flex.justify-center.gap-16 > "
#         "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
#         "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
#         "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
#     )

#     scroll_y = champion_descriptor["scroll_y"]
#     label = champion_descriptor["label"]
#     numero= champion_descriptor["numero"]

#     print(f"🧭 Scroll vers Y={scroll_y} pour {label}")
#     driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_y)
#     time.sleep(0.5)  # 👈 important

#     container = driver.find_element(By.CSS_SELECTOR, container_selector)
#     rows = container.find_elements(By.XPATH, "./div/div")

#     if not rows:
#         print("❌ Aucun champion visible après scroll")
#         return

#     row = rows[numero]  # premier visible à cet endroit
#     print(f"🖱️ Ouverture champion → {label}")

#     main_window = driver.current_window_handle

#     ActionChains(driver) \
#         .key_down(Keys.CONTROL) \
#         .click(row) \
#         .key_up(Keys.CONTROL) \
#         .perform()

#     time.sleep(2)

#     windows = driver.window_handles
#     if len(windows) < 2:
#         print("❌ Nouvel onglet non détecté")
#         return

#     new_tab = [w for w in windows if w != main_window][0]

#     for i in range(1, wait_seconds - 1):
#         print(f"⏳ {i}/{wait_seconds}s")
#         time.sleep(1)
#     driver.switch_to.window(new_tab)
#     for i in range(1, wait_seconds):
#         print(f"⏳ {i}/{wait_seconds}s")
#         time.sleep(1)
#     for i in range(1, wait_seconds):
#         print(f"⏳ {i}/{wait_seconds}s")
#         time.sleep(1)
#     # print_all_clickable_elements(driver, timeout=5)
#     # deploiement_counter_synergies(driver)
#     click_liste_complete(driver)

#     time.sleep(1.5)
#     print("sleep 1.5 avant de collecter les matchups")
#     matchups = collect_matchup(driver)
#     print(f"📦 Matchups récupérés : {len(matchups)}")

#     for i in range(1, wait_seconds + 3):
#         print(f"⏳ {i}/{wait_seconds}s")
#         time.sleep(1)
#     driver.close()
#     time.sleep(1)
#     driver.switch_to.window(main_window)
#     print("↩️ Retour liste champions")


In [6]:
def init_parse(driver, scroll_pause=1.0, scroll_step=500):

    champions_by_key = {}

    scroll_top = 0
    print("🚀 init_parse() démarré")

    while True:
        driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_top)
        time.sleep(scroll_pause)

        container_selector = (
            "#root > main > div > div.flex.justify-center.gap-16 > "
            "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
            "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
            "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
        )

        container = driver.find_element(By.CSS_SELECTOR, container_selector)
        rows = container.find_elements(By.XPATH, "./div/div")

        # driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_top)
#         time.sleep(scroll_pause)

#         container = driver.find_element(By.CSS_SELECTOR, container_selector)

#         # chaque bouton champion est dans ./div/div
#         rows = container.find_elements(By.XPATH, "./div/div")
#         print(f"📊 Lignes détectées : {len(rows)}")

        for idx, row in enumerate(rows):
            text = row.text.strip()
            if not text:
                continue

            lines = text.splitlines()
            label = lines[0]
            name = lines[1] if len(lines) > 1 else "Unknown"

            key = text  # ou (label, name)

            champions_by_key[key] = {
                "scroll_y": scroll_top,
                "label": label,
                "numero": idx,
                "name": name,
            }

        scroll_top += scroll_step
        new_height = driver.execute_script("return document.body.scrollHeight")

        if scroll_top >= new_height:
            break

    champions = list(champions_by_key.values())

    print(f"✅ {len(champions)} champions collectés (dernières occurrences)")
    return champions


In [ ]:
# from selenium.webdriver.common.by import By
# from selenium.common.exceptions import StaleElementReferenceException
# import time


# def collect_matchup(driver, max_scrolls=15, scroll_pause=0.5):
#     """
#     Parcourt les matchups Enemy après clic sur 'Liste complète'
#     et retourne une liste de dicts :
#     {
#         champ_counter_i,
#         winrate,
#         games,
#         lane_quality
#     }
#     """

#     print("🟢 Démarrage collect_matchup")

#     results = []
#     seen_champions = set()

#     # ===============================
#     # 1️⃣ Root Enemy
#     # ===============================
#     enemy_root_selector = "#radix-_r_8_-content-Enemy"
#     enemy_root = driver.find_element(By.CSS_SELECTOR, enemy_root_selector)
#     print("✅ Enemy root trouvé")

#     # ===============================
#     # 2️⃣ Conteneur des cards
#     # structure :
#     # #radix... > div > div:nth-child(1) > div
#     # ===============================
#     cards_container = enemy_root.find_element(
#         By.XPATH, "./div/div[1]/div"
#     )
#     print("✅ Conteneur de cards trouvé")

#     # ===============================
#     # 3️⃣ Scroll horizontal + collecte
#     # ===============================
#     last_scroll_left = -1

#     for scroll_i in range(max_scrolls):
#         print(f"➡️ Scroll horizontal {scroll_i + 1}/{max_scrolls}")

#         cards = cards_container.find_elements(By.XPATH, "./div")
#         print(f"🔍 {len(cards)} cards détectées")

#         for idx, card in enumerate(cards):
#             try:
#                 # ===============================
#                 # <a> (image + winrate + games)
#                 # ===============================
#                 anchor = card.find_element(By.XPATH, "./a")
#                 img = anchor.find_element(By.XPATH, ".//img")

#                 champ_name = img.get_attribute("alt").strip()
#                 if not champ_name or champ_name in seen_champions:
#                     continue

#                 seen_champions.add(champ_name)

#                 # winrate + games
#                 info_block = anchor.find_element(
#                     By.XPATH,
#                     ".//div[contains(@class,'flex') and contains(@class,'flex-col')]"
#                 )
#                 spans = info_block.find_elements(By.XPATH, ".//span")

#                 winrate = spans[0].text.strip() if len(spans) > 0 else ""
#                 games = spans[1].text.strip() if len(spans) > 1 else ""

#                 # ===============================
#                 # <button> (Bad Lane / Good Lane / Average)
#                 # ===============================
#                 lane_quality = ""
#                 try:
#                     button = card.find_element(By.XPATH, "./button")
#                     lane_quality = button.text.strip()
#                 except:
#                     pass

#                 print(
#                     f"🎯 {champ_name} | {winrate} | {games} | {lane_quality}"
#                 )

#                 results.append({
#                     "champ_counter_i": champ_name,
#                     "winrate": winrate,
#                     "games": games,
#                     "lane_quality": lane_quality,
#                 })

#             except StaleElementReferenceException:
#                 print("⚠️ StaleElement, on skip la card")
#                 continue
#             except Exception as e:
#                 print(f"❌ Erreur card {idx} → {e}")
#                 continue

#         # ===============================
#         # Scroll horizontal JS
#         # ===============================
#         current_scroll = driver.execute_script(
#             "return arguments[0].scrollLeft", cards_container
#         )

#         if current_scroll == last_scroll_left:
#             print("🛑 Fin du scroll horizontal détectée")
#             break

#         last_scroll_left = current_scroll

#         driver.execute_script(
#             "arguments[0].scrollLeft += arguments[0].offsetWidth",
#             cards_container
#         )
#         time.sleep(scroll_pause)

#     print(f"📦 Matchups collectés : {len(results)}")
#     return results


In [7]:
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException
import time

def collect_matchup(driver):
    """
    Parcourt les matchups Enemy après clic sur 'Liste complète'
    et retourne une liste de dicts :
    {
        champ_counter_i,
        winrate,
        games,
        lane_quality
    }
    """

    print("🟢 Démarrage collect_matchup")
    results = []
    seen = set()

    # ===============================
    # 1️⃣ Section Enemy
    # ===============================
    # try:
    #     enemy_root = driver.find_element(By.XPATH, '//*[@id="radix-_r_8_-content-Enemy"]')
    #     print(f"✅ Section Enemy trouvée (ID: {enemy_root.get_attribute('id')})")
    # except NoSuchElementException:
    #     print("❌ Section Enemy introuvable !")
    #     return results

    enemy_root = None
    for attempt in range(2):
        try:
            enemy_root = driver.find_element(By.XPATH, '//*[@id="radix-_r_8_-content-Enemy"]')
            print(f"✅ Section Enemy trouvée (ID: {enemy_root.get_attribute('id')})")
            break
        except NoSuchElementException:
            print(f"⏳ Tentative {attempt + 1}/2 : Section Enemy introuvable")
            time.sleep(0.8)  # petit sleep (ajuste si besoin)

    if enemy_root is None:
        print("❌ Section Enemy introuvable après 2 tentatives !")
        return results

    # ===============================
    # 2️⃣ Récupérer les deux wrappers : visible + hors écran
    # ===============================
    try:
        visible_wrapper = enemy_root.find_element(By.XPATH, "./div/div[1]/div")
        hidden_wrapper = enemy_root.find_element(By.XPATH, "./div/div[2]/div")
        print("✅ Wrappers visible et hidden trouvés")
    except NoSuchElementException:
        print("❌ Wrappers introuvables !")
        return results

    # ===============================
    # 3️⃣ Fonction de parsing des cards
    # ===============================
    def parse_cards(container):
        cards = container.find_elements(By.XPATH, "./div")
        print(f"🔍 {len(cards)} cards trouvées dans ce container")
        for idx, card in enumerate(cards):
            try:
                # ---------- <a> ----------
                anchor = card.find_element(By.XPATH, "./a")
                img = anchor.find_element(By.XPATH, ".//img")
                champ_name = img.get_attribute("alt").strip()
                if not champ_name or champ_name in seen:
                    continue
                seen.add(champ_name)

                # -------- winrate et nombre de parties --------
                info_div = anchor.find_element(By.XPATH, "./div/div[2]")  # div C
                spans = info_div.find_elements(By.XPATH, "./span")
                winrate = spans[0].text.strip() if len(spans) > 0 else ""
                games = spans[1].text.strip() if len(spans) > 1 else ""

                # -------- lane_quality --------
                lane_quality = ""
                try:
                    button = card.find_element(By.XPATH, "./button")
                    lane_quality = button.text.strip()
                except NoSuchElementException:
                    pass

                print(f"🎯 {champ_name} | Winrate: {winrate} | Games: {games} | Lane: {lane_quality}")

                results.append({
                    "champ_counter_i": champ_name,
                    "winrate": winrate,
                    "games": games,
                    "lane_quality": lane_quality
                })

            except StaleElementReferenceException:
                print("⚠️ StaleElement — skip")
            except Exception as e:
                print(f"❌ Erreur card {idx} → {e}")

    # ===============================
    # 4️⃣ Parser les cards visibles et hors écran
    # ===============================
    parse_cards(visible_wrapper)
    parse_cards(hidden_wrapper)

    print(f"📦 Matchups collectés : {len(results)}")
    return results


In [8]:
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import StaleElementReferenceException
import time


def open_champion_in_new_tab(
    driver,
    champion_descriptor: dict,
    wait_seconds: int = 5,
):
    """
    champion_descriptor = {
        "global_index": int,
        "scroll_y": int,
        "label": str
    }
    """

    container_selector = (
        "#root > main > div > div.flex.justify-center.gap-16 > "
        "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
        "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
        "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
    )

    scroll_y = champion_descriptor["scroll_y"]
    label = champion_descriptor["label"]
    numero= champion_descriptor["numero"]

    print(f"🧭 Scroll vers Y={scroll_y} pour {label}")
    driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_y)
    time.sleep(0.5)  # 👈 important

    container = driver.find_element(By.CSS_SELECTOR, container_selector)
    rows = container.find_elements(By.XPATH, "./div/div")

    if not rows:
        print("❌ Aucun champion visible après scroll")
        return

    row = rows[numero]  # premier visible à cet endroit
    print(f"🖱️ Ouverture champion → {label}")

    main_window = driver.current_window_handle

    ActionChains(driver) \
        .key_down(Keys.CONTROL) \
        .click(row) \
        .key_up(Keys.CONTROL) \
        .perform()

    time.sleep(2)

    windows = driver.window_handles
    if len(windows) < 2:
        print("❌ Nouvel onglet non détecté")
        return

    new_tab = [w for w in windows if w != main_window][0]

    for i in range(1, wait_seconds - 1):
        print(f"⏳ {i}/{wait_seconds}s")
        time.sleep(1)
    driver.switch_to.window(new_tab)
    for i in range(1, wait_seconds):
        print(f"⏳ {i}/{wait_seconds}s")
        time.sleep(1)
    for i in range(1, wait_seconds):
        print(f"⏳ {i}/{wait_seconds}s")
        time.sleep(1)
    # print_all_clickable_elements(driver, timeout=5)
    # deploiement_counter_synergies(driver)
    matchups_complets = click_liste_complete(driver)

    time.sleep(1.5)
    print("sleep 1.5 avant de collecter les matchups")


    # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    # faire une methode collect meta matchup qui dit 
    # - la lane du champ analysé
    # - la lane du champ autre
    # savoir si champ autre est synergie ou counter
    # 


    matchups = collect_matchup(driver)
    matchups["complet"] = matchups_complets

    print(f"📦 Matchups récupérés : {len(matchups)}")

    for i in range(1, wait_seconds + 3):
        print(f"⏳ {i}/{wait_seconds}s")
        time.sleep(1)
    driver.close()
    time.sleep(1)
    driver.switch_to.window(main_window)
    print("↩️ Retour liste champions")

    return matchups


In [31]:
driver = get_or_create_driver()
driver.get("https://dpm.lol/tierlist?tier=gold_plus")

🔁 Tentative de connexion à Chrome existant...
⏳ Chrome non dispo ou timeout, retry...
🚀 Timeout atteint → lancement d'un nouveau Chrome
🆕 Nouveau Chrome lancé avec debugging


In [9]:
import pandas as pd

def generate_csv_from_champions(all_champions_data, elo, server, patch, output_path="matchups_champions.csv"):
    """
    all_champions_data = [
        {
            "label": "Ahri",
            "matchups": [
                {"champ_counter_i": "LeBlanc", "winrate": "52%", "games": "120", "lane_quality": "Bad Lane"},
                ...
            ]
        },
        ...
    ]
    """
    # 1️⃣ Déterminer le nombre max de matchups
    max_matchups = max(len(champ["matchups"]) for champ in all_champions_data)

    rows = []
    print(f"🔍 Génération CSV avec {len(all_champions_data)} champions et max {max_matchups} matchups chacun")

    for champ in all_champions_data:
        print(champ)
        row = {
            "champion": champ["name"],
            "role": None,
            "ratio_games_this_role": None,
            "tier": None,
            "rank": None,
            "winrate": None,
            "pickrate": None,
            "banrate": None,
            "games_played": None,
            "matchups_complets": champ.get("matchups_complets", None),
        }

        # Ajouter les colonnes pour chaque matchup
        for i in range(max_matchups):
            #selon si on est en synergie ou matchup, on nomme le prefixe differement!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
            matchup_prefix = f"matchup_{i+1}"
            if i < len(champ["matchups"]):
                m = champ["matchups"][i]
                row[f"{matchup_prefix}_name"] = m["champ_counter_i"]
                row[f"{matchup_prefix}_winrate"] = m["winrate"]
                row[f"{matchup_prefix}_parties_observees"] = m["games"]
                row[f"{matchup_prefix}_qualite_lane"] = m["lane_quality"]
                row[f"{matchup_prefix}_role"] = None  # pour l'instant
            else:
                # Pas assez de matchups → remplir avec None
                row[f"{matchup_prefix}_name"] = None
                row[f"{matchup_prefix}_winrate"] = None
                row[f"{matchup_prefix}_parties_observees"] = None
                row[f"{matchup_prefix}_qualite_lane"] = None
                row[f"{matchup_prefix}_role"] = None

        rows.append(row)

    df = pd.DataFrame(rows)

    # Nom du fichier avec le format que tu veux
    filename = f"{output_path.replace('.csv','')}_{elo}_{server}_{patch}.csv"

    df.to_csv(filename, index=False)
    print(f"✅ CSV généré : {filename}")


In [ ]:
driver = get_or_create_driver()
driver.get("https://dpm.lol/tierlist?tier=gold_plus")

In [32]:
# la boucle mais on va remplacer la collect des champions par un click sur chacun d'entre eux.
from selenium.webdriver.common.by import By
import time

import sys
sys.stdout.flush()


# Variables pour suivre la combinaison active
elo = None
server = None
patch = None

filters_container_selector = (
    "#root > main > div > div.flex.justify-center.gap-16 > "
    "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
    "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
    "div.flex.flex-col.lg\\:flex-row.items-center.justify-between.gap-16.lg\\:gap-24.w-full > "
    "div.flex.flex-row.items-center.justify-center.gap-8.lg\\:gap-16"
)
filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)
# meta_container = driver.find_element(By.CSS_SELECTOR, meta_container_selector)

print("✅ Containers trouvés")

# =========================
# 1️⃣ Ouvrir ELO et noter la liste des boutons
# =========================
filters_container.find_element(By.XPATH, ".//button[1]").click()
time.sleep(0.5)
elo_buttons = get_last_radix_buttons()
print(f"\n🎯 ELO détectés : {[b.text for b in elo_buttons]}")

# =========================
# 2️⃣ Ouvrir SERVER et noter la liste des boutons
# =========================
filters_container.find_element(By.XPATH, ".//button[2]").click()
# main_buttons[1].click()
time.sleep(0.5)
server_buttons = get_last_radix_buttons()
print(f"🌍 SERVER détectés : {[b.text for b in server_buttons]}")

# =========================
# 3️⃣ Ouvrir PATCH et noter la liste des boutons
# =========================
filters_container.find_element(By.XPATH, ".//div/button").click()
time.sleep(0.5)
patch_buttons = get_last_radix_buttons()
print(f"🧩 PATCH détectés : {[b.text for b in patch_buttons]}")

# =========================
# BOUCLE SUR TOUTES LES COMBINAISONS
# =========================


# de elo_départ à elo_max
# for i in range(numeloDepart | 0, max(AeloFin, len(elo_buttons))):
# de elo_départ à elo_fin
# for i in range(numeloDepart | 0, min(AelohFin, len(elo_buttons))): 
for i in range(0,min(1, len(elo_buttons))):
    # 🔁 réouvrir la dropdown ELO
    filters_container.find_element(By.XPATH, ".//button[1]").click()
    time.sleep(0.7)
    elo_buttons = get_last_radix_buttons()
    elo_btn = elo_buttons[i]
    elo = elo_btn.text.strip()


    elo_btn.click()
    print(f"\n🎯 ELO [{i}] cliqué → {elo}")
    time.sleep(1)
    print("1")
    time.sleep(1)
    print("2")

    # de server_départ à server_max
    # for j in range(numServerDepart | 0, max(AServerFin, len(server_buttons))):
    # de server_départ à server_fin
    # for j in range(numServerDepart | 0, min(AServerFin, len(server_buttons))): 
    for j in range(7, min(8, len(server_buttons))):
        if ( ((j >= 3) and (j <= 10) and (j != 4) and(j!=7)) ): #7 pour LAS pour les tests
            continue  # 🔹 on skip les serveurs non désirés
        # 🔁 réouvrir la dropdown SERVER
        filters_container.find_element(By.XPATH, ".//button[2]").click()
        time.sleep(0.7)
        server_buttons = get_last_radix_buttons()
        server_btn = server_buttons[j]
        server = server_btn.text.strip()


        server_btn.click()
        print(f"  🌍 SERVER [{j}] cliqué → {server}")
        time.sleep(1)
        print("1")
        time.sleep(1)
        print("2")
        time.sleep(1)
        print("3")
        filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)


        # de patch_départ à patch_max
        # for k in range(numPatchDepart | 0, max(APatchFin, len(patch_buttons))):
        # de patch_départ à patch_fin
        # for k in range(numPatchDepart | 0, min(APatchFin, len(patch_buttons))): 
        for k in range(5, min(6, len(patch_buttons))):  # 🔹 on limite à 3 itérations pour tester          
            # 🔁 réouvrir la dropdown PATCH
            filters_container.find_element(By.XPATH, ".//div/button").click()
            time.sleep(1)

            # 🔹 récupérer à nouveau les boutons PATCH pour éviter StaleElementReference
            patch_buttons = get_last_radix_buttons()
            time.sleep(1)
            print("click sur les patchs")
            print(f"🧩 PATCH mis à jour : {[b.text for b in patch_buttons]}")
            patch_btn = patch_buttons[k]

            # 🔹 cliquer sur le kème bouton
            patch = patch_btn.text.strip()
            patch_btn.click()
            time.sleep(0.7)

            print(f"    🧩 PATCH [{k}] cliqué → {patch}")

            # ✅ COMBINAISON ACTIVE
            print(f"    ✅ COMBINAISON ACTIVE : ELO={elo}, SERVER={server}, PATCH={patch}")
            time.sleep(1)
            print("1")
            time.sleep(1)
            print("2")
            time.sleep(1)
            print("3")
            
            filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)
            # Ici tu peux lancer ton scraping pour la combinaison active
            # df = scrape_champions(driver, elo, server, patch)

            # champion_buttons = init_parse(driver)

            # print(f"\n🎯 {len(champion_buttons)} champions trouvés\n")

            # for i, btn in enumerate(champion_buttons[:5]):
            #     # print(f"[{i}] {btn.text.splitlines()[0]}")
            #     open_champion_in_new_tab(driver, champion_index=i, wait_seconds=5)


            champions = init_parse(driver)
            all_champions_data = []  # liste qui va contenir les objets champion

            # for champ in champions[:5]:  
            for champ in champions[:2]:  # les 5 derniers pour tester
                print(champ)
                matchups = open_champion_in_new_tab(driver, champ, wait_seconds=5)
                champ_data = {
                    "name": champ["name"],
                    "matchups": matchups
                }
                all_champions_data.append(champ_data)

            generate_csv_from_champions(
                all_champions_data,
                elo=elo,
                server=server,
                patch=patch
            )    

            # for champ in champions[(len(champions)-2):]:  # les 5 derniers pour tester
            #     print(champ)
            #     open_champion_in_new_tab(driver, champ, wait_seconds=5)

            driver.execute_script("window.scrollTo(0, arguments[0]);", 0)




✅ Containers trouvés

🎯 ELO détectés : ['Challenger', 'Grandmaster', 'Master+', 'Master', 'Diamond+', 'Diamond', 'Emerald+', 'Emerald', 'Platinum+', 'Platinum', 'Gold+', 'Gold', 'Silver+', 'Bronze', 'Iron', 'TOUT']
🌍 SERVER détectés : ['EUW', 'KR', 'NA', 'BR', 'EUNE', 'JP', 'LAN', 'LAS', 'OCE', 'RU', 'TR', 'VN', 'TOUT']
🧩 PATCH détectés : ['7days', '14days', '30days', '16.3', '16.2', '16.1', '15.24', '15.23', '15.22']

🎯 ELO [0] cliqué → Challenger
1
2
  🌍 SERVER [7] cliqué → LAS
1
2
3
click sur les patchs
🧩 PATCH mis à jour : ['7days', '14days', '30days', '16.3', '16.2', '16.1', '15.24', '15.23', '15.22']
    🧩 PATCH [5] cliqué → 16.1
    ✅ COMBINAISON ACTIVE : ELO=Challenger, SERVER=LAS, PATCH=16.1
1
2
3
🚀 init_parse() démarré
✅ 4 champions collectés (dernières occurrences)
{'scroll_y': 1000, 'label': '1', 'numero': 0, 'name': 'Aphelios'}
🧭 Scroll vers Y=1000 pour 1
🖱️ Ouverture champion → 1
⏳ 1/5s
⏳ 2/5s
⏳ 3/5s
⏳ 1/5s
⏳ 2/5s
⏳ 3/5s
⏳ 4/5s
⏳ 1/5s
⏳ 2/5s
⏳ 3/5s
⏳ 4/5s
🟢 Recherche du b

In [ ]:
driver = get_or_create_driver()
driver.get("https://dpm.lol/tierlist?tier=gold_plus")

In [ ]:
#sauvegarde de la boucle principale avant de devoir enregistrer les infos par champion

# # la boucle mais on va remplacer la collect des champions par un click sur chacun d'entre eux.
# from selenium.webdriver.common.by import By
# import time

# import sys
# sys.stdout.flush()


# # Variables pour suivre la combinaison active
# elo = None
# server = None
# patch = None

# filters_container_selector = (
#     "#root > main > div > div.flex.justify-center.gap-16 > "
#     "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
#     "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
#     "div.flex.flex-col.lg\\:flex-row.items-center.justify-between.gap-16.lg\\:gap-24.w-full > "
#     "div.flex.flex-row.items-center.justify-center.gap-8.lg\\:gap-16"
# )
# filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)
# # meta_container = driver.find_element(By.CSS_SELECTOR, meta_container_selector)

# print("✅ Containers trouvés")

# # =========================
# # 1️⃣ Ouvrir ELO et noter la liste des boutons
# # =========================
# filters_container.find_element(By.XPATH, ".//button[1]").click()
# time.sleep(0.5)
# elo_buttons = get_last_radix_buttons()
# print(f"\n🎯 ELO détectés : {[b.text for b in elo_buttons]}")

# # =========================
# # 2️⃣ Ouvrir SERVER et noter la liste des boutons
# # =========================
# filters_container.find_element(By.XPATH, ".//button[2]").click()
# # main_buttons[1].click()
# time.sleep(0.5)
# server_buttons = get_last_radix_buttons()
# print(f"🌍 SERVER détectés : {[b.text for b in server_buttons]}")

# # =========================
# # 3️⃣ Ouvrir PATCH et noter la liste des boutons
# # =========================
# filters_container.find_element(By.XPATH, ".//div/button").click()
# time.sleep(0.5)
# patch_buttons = get_last_radix_buttons()
# print(f"🧩 PATCH détectés : {[b.text for b in patch_buttons]}")

# # =========================
# # BOUCLE SUR TOUTES LES COMBINAISONS
# # =========================


# # de elo_départ à elo_max
# # for i in range(numeloDepart | 0, max(AeloFin, len(elo_buttons))):
# # de elo_départ à elo_fin
# # for i in range(numeloDepart | 0, min(AelohFin, len(elo_buttons))): 
# for i in range(0,min(1, len(elo_buttons))):
#     # 🔁 réouvrir la dropdown ELO
#     filters_container.find_element(By.XPATH, ".//button[1]").click()
#     time.sleep(0.7)
#     elo_buttons = get_last_radix_buttons()
#     elo_btn = elo_buttons[i]
#     elo = elo_btn.text.strip()


#     elo_btn.click()
#     print(f"\n🎯 ELO [{i}] cliqué → {elo}")
#     time.sleep(1)
#     print("1")
#     time.sleep(1)
#     print("2")

#     # de server_départ à server_max
#     # for j in range(numServerDepart | 0, max(AServerFin, len(server_buttons))):
#     # de server_départ à server_fin
#     # for j in range(numServerDepart | 0, min(AServerFin, len(server_buttons))): 
#     for j in range(7, min(8, len(server_buttons))):
#         if ( ((j >= 3) and (j <= 10) and (j != 4) and(j!=7)) ): #7 pour LAS pour les tests
#             continue  # 🔹 on skip les serveurs non désirés
#         # 🔁 réouvrir la dropdown SERVER
#         filters_container.find_element(By.XPATH, ".//button[2]").click()
#         time.sleep(0.7)
#         server_buttons = get_last_radix_buttons()
#         server_btn = server_buttons[j]
#         server = server_btn.text.strip()


#         server_btn.click()
#         print(f"  🌍 SERVER [{j}] cliqué → {server}")
#         time.sleep(1)
#         print("1")
#         time.sleep(1)
#         print("2")
#         time.sleep(1)
#         print("3")
#         filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)

#         # de patch_départ à patch_max
#         # for k in range(numPatchDepart | 0, max(APatchFin, len(patch_buttons))):
#         # de patch_départ à patch_fin
#         # for k in range(numPatchDepart | 0, min(APatchFin, len(patch_buttons))): 
#         for k in range(5, min(6, len(patch_buttons))):  # 🔹 on limite à 3 itérations pour tester          
#             # 🔁 réouvrir la dropdown PATCH
#             filters_container.find_element(By.XPATH, ".//div/button").click()
#             time.sleep(1)

#             # 🔹 récupérer à nouveau les boutons PATCH pour éviter StaleElementReference
#             patch_buttons = get_last_radix_buttons()
#             time.sleep(1)
#             print("click sur les patchs")
#             print(f"🧩 PATCH mis à jour : {[b.text for b in patch_buttons]}")
#             patch_btn = patch_buttons[k]

#             # 🔹 cliquer sur le kème bouton
#             patch = patch_btn.text.strip()
#             patch_btn.click()
#             time.sleep(0.7)

#             print(f"    🧩 PATCH [{k}] cliqué → {patch}")

#             # ✅ COMBINAISON ACTIVE
#             print(f"    ✅ COMBINAISON ACTIVE : ELO={elo}, SERVER={server}, PATCH={patch}")
#             time.sleep(1)
#             print("1")
#             time.sleep(1)
#             print("2")
#             time.sleep(1)
#             print("3")
            
#             filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)
#             # Ici tu peux lancer ton scraping pour la combinaison active
#             # df = scrape_champions(driver, elo, server, patch)

#             # champion_buttons = init_parse(driver)

#             # print(f"\n🎯 {len(champion_buttons)} champions trouvés\n")

#             # for i, btn in enumerate(champion_buttons[:5]):
#             #     # print(f"[{i}] {btn.text.splitlines()[0]}")
#             #     open_champion_in_new_tab(driver, champion_index=i, wait_seconds=5)


#             champions = init_parse(driver)

            

#             # for champ in champions[:5]:  
#             for champ in champions[:2]:  # les 5 derniers pour tester
#                 print(champ)
#                 open_champion_in_new_tab(driver, champ, wait_seconds=5)
#             for champ in champions[(len(champions)-2):]:  # les 5 derniers pour tester
#                 print(champ)
#                 open_champion_in_new_tab(driver, champ, wait_seconds=5)

#             driver.execute_script("window.scrollTo(0, arguments[0]);", 0)




In [ ]:
# from selenium.webdriver.common.by import By
# from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException
# import time

# def collect_matchup_standalone(driver):
#     """
#     Parcourt les matchups Enemy (cards visibles et hors écran)
#     et retourne une liste de dicts :
#     {
#         champ_counter_i,
#         winrate,
#         games,
#         lane_quality
#     }
#     """

#     print("\n🟢 collect_matchup_standalone — démarrage")
#     results = []
#     seen = set()

#     # ===============================
#     # 1️⃣ Section Enemy
#     # ===============================
#     try:
#         enemy_root = driver.find_element(By.XPATH, '//*[@id="radix-_r_8_-content-Enemy"]')
#         print(f"✅ Section Enemy trouvée (ID: {enemy_root.get_attribute('id')})")
#     except NoSuchElementException:
#         print("❌ Section Enemy introuvable !")
#         return results

#     # ===============================
#     # 2️⃣ Récupérer les deux wrappers : visible + hors écran
#     # ===============================
#     try:
#         visible_wrapper = enemy_root.find_element(By.XPATH, "./div/div[1]/div")
#         hidden_wrapper = enemy_root.find_element(By.XPATH, "./div/div[2]/div")
#         print("✅ Wrappers visible et hidden trouvés")
#     except NoSuchElementException:
#         print("❌ Wrappers introuvables !")
#         return results

#     # ===============================
#     # 3️⃣ Fonction de parsing des cards
#     # ===============================
#     def parse_cards(container):
#         cards = container.find_elements(By.XPATH, "./div")
#         print(f"🔍 {len(cards)} cards trouvées dans ce container")
#         for idx, card in enumerate(cards):
#             try:
#                 anchor = card.find_element(By.XPATH, "./a")
#                 img = anchor.find_element(By.XPATH, ".//img")
#                 champ_name = img.get_attribute("alt").strip()
#                 if not champ_name or champ_name in seen:
#                     continue
#                 seen.add(champ_name)

#                 # -------- winrate et nombre de parties --------
#                 info_block = anchor.find_element(By.XPATH, ".//div[contains(@class,'flex-col')]")
#                 spans = info_block.find_elements(By.XPATH, ".//span")
#                 winrate = spans[0].text.strip() if len(spans) > 0 else ""
#                 games = spans[1].text.strip() if len(spans) > 1 else ""

#                 # -------- lane_quality --------
#                 lane_quality = ""
#                 try:
#                     button = card.find_element(By.XPATH, "./button")
#                     lane_quality = button.text.strip()
#                 except NoSuchElementException:
#                     pass

#                 print(f"🎯 {champ_name} | Winrate: {winrate} | Games: {games} | Lane: {lane_quality}")

#                 results.append({
#                     "champ_counter_i": champ_name,
#                     "winrate": winrate,
#                     "games": games,
#                     "lane_quality": lane_quality
#                 })

#             except StaleElementReferenceException:
#                 print("⚠️ StaleElement — skip")
#             except Exception as e:
#                 print(f"❌ Erreur card {idx} → {e}")

#     # ===============================
#     # 4️⃣ Parser les cards visibles et hors écran
#     # ===============================
#     parse_cards(visible_wrapper)
#     parse_cards(hidden_wrapper)

#     print(f"\n📦 Total matchups collectés : {len(results)}")
#     return results


In [37]:
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException
import time

def collect_matchup_standalone(driver):
    """
    Parcourt les matchups Enemy (cards visibles et hors écran)
    et retourne une liste de dicts :
    {
        champ_counter_i,
        winrate,
        games,
        lane_quality
    }
    """

    print("\n🟢 collect_matchup_standalone — démarrage")
    results = []
    seen = set()

    # ===============================
    # 1️⃣ Section Enemy
    # ===============================
    try:
        enemy_root = driver.find_element(By.XPATH, '//*[@id="radix-_r_8_-content-Enemy"]')
        print(f"✅ Section Enemy trouvée (ID: {enemy_root.get_attribute('id')})")
    except NoSuchElementException:
        print("❌ Section Enemy introuvable !")
        return results

    # ===============================
    # 2️⃣ Récupérer les deux wrappers : visible + hors écran
    # ===============================
    try:
        visible_wrapper = enemy_root.find_element(By.XPATH, "./div/div[1]/div")
        hidden_wrapper = enemy_root.find_element(By.XPATH, "./div/div[2]/div")
        print("✅ Wrappers visible et hidden trouvés")
    except NoSuchElementException:
        print("❌ Wrappers introuvables !")
        return results

    # ===============================
    # 3️⃣ Fonction de parsing des cards
    # ===============================
    def parse_cards(container):
        cards = container.find_elements(By.XPATH, "./div")
        print(f"🔍 {len(cards)} cards trouvées dans ce container")
        for idx, card in enumerate(cards):
            try:
                anchor = card.find_element(By.XPATH, "./a")
                img = anchor.find_element(By.XPATH, ".//img")
                champ_name = img.get_attribute("alt").strip()
                if not champ_name or champ_name in seen:
                    continue
                seen.add(champ_name)

                # -------- winrate et nombre de parties --------
                # div C
                info_div = anchor.find_element(By.XPATH, "./div/div[2]")
                spans = info_div.find_elements(By.XPATH, "./span")
                winrate = spans[0].text.strip() if len(spans) > 0 else ""
                games = spans[1].text.strip() if len(spans) > 1 else ""

                # -------- lane_quality --------
                lane_quality = ""
                try:
                    button = card.find_element(By.XPATH, "./button")
                    lane_quality = button.text.strip()
                except NoSuchElementException:
                    pass

                print(f"🎯 {champ_name} | Winrate: {winrate} | Games: {games} | Lane: {lane_quality}")

                results.append({
                    "champ_counter_i": champ_name,
                    "winrate": winrate,
                    "games": games,
                    "lane_quality": lane_quality
                })

            except StaleElementReferenceException:
                print("⚠️ StaleElement — skip")
            except Exception as e:
                print(f"❌ Erreur card {idx} → {e}")

    # ===============================
    # 4️⃣ Parser les cards visibles et hors écran
    # ===============================
    parse_cards(visible_wrapper)
    parse_cards(hidden_wrapper)

    print(f"\n📦 Total matchups collectés : {len(results)}")
    return results


In [38]:
driver = get_or_create_driver()
driver.get("https://dpm.lol/champions/Milio/build?lane=utility&tier=gold_plus&platform=la2&timeframe=15.24")

🔁 Tentative de connexion à Chrome existant...
⏳ Chrome non dispo ou timeout, retry...
🚀 Timeout atteint → lancement d'un nouveau Chrome
🆕 Nouveau Chrome lancé avec debugging


In [44]:
matchups = collect_matchup_standalone(driver)

for m in matchups:
    print(m)


🟢 collect_matchup_standalone — démarrage
✅ Section Enemy trouvée (ID: radix-_r_8_-content-Enemy)
✅ Wrappers visible et hidden trouvés
🔍 5 cards trouvées dans ce container
🎯 Mel | Winrate: 60.58% | Games: 1677 | Lane: Average
🎯 Malphite | Winrate: 60.31% | Games: 320 | Lane: Bad Lane
🎯 Shen | Winrate: 59.20% | Games: 250 | Lane: Bad Lane
🎯 Sylas | Winrate: 58.87% | Games: 248 | Lane: Average
🎯 Ashe | Winrate: 58.33% | Games: 336 | Lane: Good Lane
🔍 5 cards trouvées dans ce container
🎯 Velkoz | Winrate: 50.13% | Games: 1179 | Lane: Bad Lane
🎯 Sona | Winrate: 49.83% | Games: 869 | Lane: Good Lane
🎯 Karma | Winrate: 49.68% | Games: 3424 | Lane: Average
🎯 Janna | Winrate: 49.47% | Games: 1403 | Lane: Good Lane
🎯 Zyra | Winrate: 49.25% | Games: 1460 | Lane: Bad Lane

📦 Total matchups collectés : 10
{'champ_counter_i': 'Mel', 'winrate': '60.58%', 'games': '1677', 'lane_quality': 'Average'}
{'champ_counter_i': 'Malphite', 'winrate': '60.31%', 'games': '320', 'lane_quality': 'Bad Lane'}
{'cham

In [ ]:
# #code pour spot tous les boutons
#         # buttons = driver.find_elements(By.TAG_NAME, "button")
#         # print(f"🔍 {len(buttons)} boutons trouvés sur la page")

#         # for i, btn in enumerate(buttons):
#         #     try:
#         #         btn_text = btn.text.strip()
#         #         btn_text_clean = " ".join(btn_text.split())

#         #         # XPath absolu (debug)
#         #         btn_xpath = driver.execute_script(
#         #             """
#         #             function getXPath(element) {
#         #                 if (element.id !== '')
#         #                     return '//*[@id="' + element.id + '"]';
#         #                 if (element === document.body)
#         #                     return '/html/body';

#         #                 let ix = 0;
#         #                 let siblings = element.parentNode.childNodes;
#         #                 for (let i = 0; i < siblings.length; i++) {
#         #                     let sibling = siblings[i];
#         #                     if (sibling === element)
#         #                         return getXPath(element.parentNode) + '/' +
#         #                             element.tagName.toLowerCase() + '[' + (ix + 1) + ']';
#         #                     if (sibling.nodeType === 1 && sibling.tagName === element.tagName)
#         #                         ix++;
#         #                 }
#         #             }
#         #             return getXPath(arguments[0]);
#         #             """,
#         #             btn,
#         #         )

#         #         print(f"[{i}] → '{btn_text_clean}' | XPath: {btn_xpath}")

#         #         if "+ Liste complète" in btn_text_clean:
#         #             print("🎯 Bouton '+ Liste complète' détecté, tentative de clic...")

#         #             driver.execute_script(
#         #                 "arguments[0].scrollIntoView({block: 'center'});",
#         #                 btn,
#         #             )
#         #             time.sleep(0.75)
#         #             btn.click()

#         #             print("🖱️ Bouton '+ Liste complète' cliqué avec succès", flush=True)
#         #             time.sleep(1)
#         #             return True  # ✅ succès immédiat

#         #     except StaleElementReferenceException:
#         #         print(f"⚠️ Bouton [{i}] devenu obsolète (DOM mis à jour)")
#         #     except Exception as e:
#         #         print(f"⚠️ Erreur sur le bouton [{i}] : {e}")


# #code pour transformer l'image de supp en supp



# #code qui doit permettre de qualifier les matchups : 
# # general : 
# # - lane du champ analysé
# # - pourcentage de games à ce role
# # - tier, rang, WR, PR, BR, nb games analysées

# # propre aux itérations : 
# # - matchup / synergie
# # - lane du champ autre

# from selenium.webdriver.common.by import By
# from selenium.common.exceptions import StaleElementReferenceException
# import time


# def pre_collect_matchup_standalone(driver, synergy: bool, lane: str):
#     """
#     Premier jet :
#     - Ne clique sur rien
#     - Liste et log tous les boutons visibles
#     - Permet d'identifier :
#         - bouton Synergies
#         - bouton Matchup
#         - boutons de lane (top/jun/mid/adc/supp)
#     """

#     print("🟢 pre_collect_matchup_standalone() — MODE DEBUG")
#     print(f"➡️ synergy = {synergy}")
#     print(f"➡️ lane = {lane}")
#     print("🔍 Scan de tous les boutons visibles à l'écran...\n")

#     time.sleep(1)  # petit buffer DOM

#     buttons = driver.find_elements(By.TAG_NAME, "button")
#     print(f"🔢 {len(buttons)} boutons trouvés\n")

#     for i, btn in enumerate(buttons):
#         try:
#             text_raw = btn.text.strip()
#             text_clean = " ".join(text_raw.split())

#             aria_label = btn.get_attribute("aria-label")
#             data_state = btn.get_attribute("data-state")
#             class_name = btn.get_attribute("class")

#             # XPath de debug
#             xpath = driver.execute_script(
#                 """
#                 function getXPath(el) {
#                     if (el.id)
#                         return '//*[@id="' + el.id + '"]';
#                     if (el === document.body)
#                         return '/html/body';

#                     let ix = 0;
#                     let siblings = el.parentNode.childNodes;
#                     for (let i = 0; i < siblings.length; i++) {
#                         let sibling = siblings[i];
#                         if (sibling === el)
#                             return getXPath(el.parentNode) + '/' +
#                                    el.tagName.toLowerCase() + '[' + (ix + 1) + ']';
#                         if (sibling.nodeType === 1 && sibling.tagName === el.tagName)
#                             ix++;
#                     }
#                 }
#                 return getXPath(arguments[0]);
#                 """,
#                 btn,
#             )

#             print(f"[{i}]")
#             print(f"   📝 text        : '{text_clean}'")
#             print(f"   🏷️ aria-label : {aria_label}")
#             print(f"   📦 data-state : {data_state}")
#             print(f"   🎨 class      : {class_name}")
#             print(f"   🧭 xpath      : {xpath}")
#             print("-" * 60)

#         except StaleElementReferenceException:
#             print(f"[{i}] ⚠️ Bouton obsolète (DOM mis à jour)")
#         except Exception as e:
#             print(f"[{i}] ❌ Erreur lors de l'inspection : {e}")

#     print("\n✅ Scan terminé — partage-moi les logs et je t'identifie les bons boutons 👌")



In [63]:
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException

def collect_champion_lane_stats(driver) -> dict:
    print("🟢 Collecte des stats champion / lane")

    stats = {
        "tier": None,
        "rank": None,
        "winrate": None,
        "pickrate": None,
        "banrate": None,
        "games": None,
    }

    try:
        container = driver.find_element(
            By.XPATH,
            '//*[@id="root"]/main/div/div[3]/div[2]/div/div[2]/div[1]'
        )
    except NoSuchElementException:
        print("❌ Container stats introuvable")
        return stats

    print("✅ Container stats trouvé")
    print("🎨 class :", container.get_attribute("class"))

    spans = container.find_elements(By.TAG_NAME, "span")
    print(f"🔢 {len(spans)} spans trouvés\n")

    KNOWN_KEYS = {
        "niveau": "tier",
        "rang": "rank",
        "winrate": "winrate",
        "pickrate": "pickrate",
        "banrate": "banrate",
        "parties": "games",
    }

    for idx, span in enumerate(spans):
        raw_text = span.text.strip()
        if not raw_text:
            continue

        print(f"[Span {idx}] → '{raw_text}'")

        lower = raw_text.lower()

        for label, key in KNOWN_KEYS.items():
            if label in lower:
                # ne pas écraser une valeur déjà trouvée
                if stats[key] is not None:
                    continue

                # extraction valeur
                value = (
                    raw_text
                    .replace(label, "")
                    .replace("\n", " ")
                    .strip()
                )

                # ignorer les spans "label only"
                if not value:
                    continue

                # suppression de l'espace insécable pour "games"
                if key == "games":
                    value = value.replace("\u202f", "").replace(" ", "")

                stats[key] = value
                print(f"✅ {key} détecté → '{value}'")

    print("-" * 60)
    print("✅ Stats collectées :", stats)
    return stats


In [64]:
collect_champion_lane_stats(driver)

🟢 Collecte des stats champion / lane
✅ Container stats trouvé
🎨 class : max-w-6xl bg-purple-200/[0.04] overflow-hidden border-t border-black-0 border-opacity-10 pt-4 h-fit rounded-md grid grid-cols-3 lg:grid-cols-6 items-center justify-center gap-y-12 text-bl lg:text-hs text-black-100 font-semibold text-center 
🔢 17 spans trouvés

[Span 0] → 'S+
niveau'
✅ tier détecté → 'S+'
[Span 1] → 'niveau'
[Span 2] → '1
/ 37
rang'
✅ rank détecté → '1 / 37'
[Span 3] → '/ 37'
[Span 4] → 'rang'
[Span 5] → '51.6
%
winrate'
✅ winrate détecté → '51.6 %'
[Span 6] → '%'
[Span 7] → 'winrate'
[Span 8] → '10.2
%
pickrate'
✅ pickrate détecté → '10.2 %'
[Span 9] → '%'
[Span 10] → 'pickrate'
[Span 11] → '19.5
%
banrate'
✅ banrate détecté → '19.5 %'
[Span 12] → '%'
[Span 13] → 'banrate'
[Span 14] → '67 241
parties'
✅ games détecté → '67241'
[Span 15] → 'parties'
------------------------------------------------------------
✅ Stats collectées : {'tier': 'S+', 'rank': '1 / 37', 'winrate': '51.6 %', 'pickrate': '10.

{'tier': 'S+',
 'rank': '1 / 37',
 'winrate': '51.6 %',
 'pickrate': '10.2 %',
 'banrate': '19.5 %',
 'games': '67241'}

In [15]:
from selenium.webdriver.common.by import By
from selenium.common.exceptions import StaleElementReferenceException
import time


def pre_collect_matchup_standalone(driver, synergy: bool, lane: str):
    print("🟢 pre_collect_matchup_standalone() — MODE DEBUG SVG")
    print(f"➡️ synergy = {synergy}")
    print(f"➡️ lane = {lane}")
    print("🔍 Scan de tous les boutons visibles à l'écran...\n")

    time.sleep(1)

    buttons = driver.find_elements(By.TAG_NAME, "button")
    print(f"🔢 {len(buttons)} boutons trouvés\n")

    for i, btn in enumerate(buttons):
        try:
            text_raw = btn.text.strip()
            text_clean = " ".join(text_raw.split())

            aria_label = btn.get_attribute("aria-label")
            data_state = btn.get_attribute("data-state")
            class_name = btn.get_attribute("class")

            # XPath de debug
            xpath = driver.execute_script(
                """
                function getXPath(el) {
                    if (el.id)
                        return '//*[@id="' + el.id + '"]';
                    if (el === document.body)
                        return '/html/body';

                    let ix = 0;
                    let siblings = el.parentNode.childNodes;
                    for (let i = 0; i < siblings.length; i++) {
                        let sibling = siblings[i];
                        if (sibling === el)
                            return getXPath(el.parentNode) + '/' +
                                   el.tagName.toLowerCase() + '[' + (ix + 1) + ']';
                        if (sibling.nodeType === 1 && sibling.tagName === el.tagName)
                            ix++;
                    }
                }
                return getXPath(arguments[0]);
                """,
                btn,
            )

            # 🔽 récupération des balises enfants + SVG
            children_and_svg = driver.execute_script(
                """
                const el = arguments[0];
                const children = el.children;

                let tags = [];
                let svgs = [];

                for (let i = 0; i < children.length; i++) {
                    const tag = children[i].tagName.toLowerCase();
                    tags.push(tag);

                    if (tag === 'svg') {
                        svgs.push(children[i].outerHTML);
                    }
                }

                return {
                    tags: tags,
                    svgs: svgs
                };
                """,
                btn,
            )

            print(f"[{i}]")
            print(f"   📝 text        : '{text_clean}'")
            print(f"   🏷️ aria-label : {aria_label}")
            print(f"   📦 data-state : {data_state}")
            print(f"   🎨 class      : {class_name}")
            print(f"   🧭 xpath      : {xpath}")

            if children_and_svg["tags"]:
                print(f"   🧩 children   : {children_and_svg['tags']}")
            else:
                print("   🧩 children   : aucun")

            # 🔥 affichage du SVG s'il existe
            if children_and_svg["svgs"]:
                for idx, svg in enumerate(children_and_svg["svgs"]):
                    print(f"   🎨 SVG #{idx + 1} :")
                    print(svg)
            else:
                print("   🎨 SVG        : aucun")

            print("-" * 80)

        except StaleElementReferenceException:
            print(f"[{i}] ⚠️ Bouton obsolète (DOM mis à jour)")
        except Exception as e:
            print(f"[{i}] ❌ Erreur lors de l'inspection : {e}")

    print("\n✅ Scan terminé")
    print("👉 Envoie-moi les blocs des boutons Synergy / Matchup / Lanes")


In [16]:
pre_collect_matchup_standalone(driver, synergy=True, lane="top")

🟢 pre_collect_matchup_standalone() — MODE DEBUG SVG
➡️ synergy = True
➡️ lane = top
🔍 Scan de tous les boutons visibles à l'écran...

🔢 55 boutons trouvés

[0]
   📝 text        : 'Tierlist & Builds'
   🏷️ aria-label : None
   📦 data-state : closed
   🎨 class      : group inline-flex h-4 w-max items-center justify-center rounded-md bg-background px-4 py-2 text-bm font-medium transition-colors hover:bg-accent hover:text-accent-foreground focus:bg-accent focus:text-accent-foreground focus:outline-none disabled:pointer-events-none disabled:opacity-50 data-[state=open]:text-accent-foreground data-[state=open]:bg-accent/50 data-[state=open]:hover:bg-accent data-[state=open]:focus:bg-accent group
   🧭 xpath      : //*[@id="radix-_R_59alb_-trigger-radix-_R_5l9alb_"]
   🧩 children   : ['a', 'svg']
   🎨 SVG #1 :
<svg xmlns="http://www.w3.org/2000/svg" width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" class

In [46]:
pre_collect_matchup_standalone(driver, synergy=True, lane="top")

🟢 pre_collect_matchup_standalone() — MODE DEBUG SVG
➡️ synergy = True
➡️ lane = top
🔍 Scan de tous les boutons visibles à l'écran...

🔢 55 boutons trouvés

[0]
   📝 text        : ''
   🏷️ aria-label : None
   📦 data-state : closed
   🎨 class      : group inline-flex h-4 w-max items-center justify-center rounded-md bg-background px-4 py-2 text-bm font-medium transition-colors hover:bg-accent hover:text-accent-foreground focus:bg-accent focus:text-accent-foreground focus:outline-none disabled:pointer-events-none disabled:opacity-50 data-[state=open]:text-accent-foreground data-[state=open]:bg-accent/50 data-[state=open]:hover:bg-accent data-[state=open]:focus:bg-accent group
   🧭 xpath      : //*[@id="radix-_R_59alb_-trigger-radix-_R_5l9alb_"]
   🧩 children   : ['a', 'svg']
   🎨 SVG #1 :
<svg xmlns="http://www.w3.org/2000/svg" width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" class="lucide lucide-c

In [49]:
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, WebDriverException
import time

def click_matchup_or_synergy(driver, matchup: bool) -> bool:
    target_name = "MATCHUPS" if matchup else "SYNERGIES"
    aria_key = "Enemy" if matchup else "Ally"

    print(f"🟢 Sélection de l'onglet {target_name}", flush=True)

    xpath = f"//button[@role='tab' and contains(@aria-controls, '{aria_key}')]"

    try:
        button = driver.find_element(By.XPATH, xpath)

        data_state = button.get_attribute("data-state")
        print(f"🔍 {target_name} data-state = {data_state}")

        if data_state == "active":
            print(f"ℹ️ Onglet {target_name} déjà actif")
            return False

        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});",
            button
        )
        time.sleep(0.3)

        button.click()
        print(f"🖱️ Onglet {target_name} cliqué", flush=True)
        time.sleep(0.7)

        return True

    except NoSuchElementException:
        print(f"❌ Onglet {target_name} introuvable (aria-controls)", flush=True)
        return False

    except WebDriverException as e:
        print(f"❌ Erreur Selenium sur {target_name} : {e}", flush=True)
        return False


In [50]:
click_matchup_or_synergy(driver, matchup=True)
time.sleep(3)
click_matchup_or_synergy(driver, matchup=False)
time.sleep(3)
click_matchup_or_synergy(driver, matchup=True)
time.sleep(3)
click_matchup_or_synergy(driver, matchup=False)

🟢 Sélection de l'onglet MATCHUPS
🔍 MATCHUPS data-state = active
ℹ️ Onglet MATCHUPS déjà actif
🟢 Sélection de l'onglet SYNERGIES
🔍 SYNERGIES data-state = inactive
🖱️ Onglet SYNERGIES cliqué
🟢 Sélection de l'onglet MATCHUPS
🔍 MATCHUPS data-state = inactive
🖱️ Onglet MATCHUPS cliqué
🟢 Sélection de l'onglet SYNERGIES
🔍 SYNERGIES data-state = inactive
🖱️ Onglet SYNERGIES cliqué


True

In [17]:
from selenium.webdriver.common.by import By
from selenium.common.exceptions import StaleElementReferenceException
import time


def debug_links_with_svg(driver):
    print("🟣 DEBUG <a> + <svg> — MODE EXPLORATION")
    time.sleep(1)

    links = driver.find_elements(By.TAG_NAME, "a")
    print(f"🔢 {len(links)} balises <a> trouvées\n")

    for i, link in enumerate(links):
        try:
            # récup svg enfants
            svgs = link.find_elements(By.TAG_NAME, "svg")
            if not svgs:
                continue  # on ignore les <a> sans svg

            class_name = link.get_attribute("class")
            href = link.get_attribute("href")

            # span adjacent (souvent frère)
            span_text = driver.execute_script(
                """
                const el = arguments[0];
                const span = el.nextElementSibling;
                return span && span.tagName.toLowerCase() === 'span'
                    ? span.innerText.trim()
                    : null;
                """,
                link,
            )

            xpath = driver.execute_script(
                """
                function getXPath(el) {
                    if (el.id)
                        return '//*[@id="' + el.id + '"]';
                    if (el === document.body)
                        return '/html/body';

                    let ix = 0;
                    let siblings = el.parentNode.childNodes;
                    for (let i = 0; i < siblings.length; i++) {
                        let sibling = siblings[i];
                        if (sibling === el)
                            return getXPath(el.parentNode) + '/' +
                                   el.tagName.toLowerCase() + '[' + (ix + 1) + ']';
                        if (sibling.nodeType === 1 && sibling.tagName === el.tagName)
                            ix++;
                    }
                }
                return getXPath(arguments[0]);
                """,
                link,
            )

            print(f"[A-{i}]")
            print(f"   🔗 href       : {href}")
            print(f"   🎨 class      : {class_name}")
            print(f"   🧭 xpath      : {xpath}")
            print(f"   🏷️ span text : {span_text}")

            for idx, svg in enumerate(svgs):
                print(f"   🎨 SVG #{idx + 1}:")
                print(svg.get_attribute("outerHTML"))

            print("-" * 80)

        except StaleElementReferenceException:
            print(f"[A-{i}] ⚠️ élément obsolète")
        except Exception as e:
            print(f"[A-{i}] ❌ erreur : {e}")

    print("\n✅ Scan <a> terminé")


In [18]:
debug_links_with_svg(driver)

🟣 DEBUG <a> + <svg> — MODE EXPLORATION
🔢 61 balises <a> trouvées

[A-8]
   🔗 href       : https://dpm.lol/champions/Braum/build
   🎨 class      : 
   🧭 xpath      : //*[@id="root"]/main[1]/div[1]/div[1]/div[1]/div[1]/div[1]/a[1]
   🏷️ span text : None
   🎨 SVG #1:
<svg xmlns="http://www.w3.org/2000/svg" width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" class="lucide lucide-zap size-12 lg:size-16"><path d="M4 14a1 1 0 0 1-.78-1.63l9.9-10.2a.5.5 0 0 1 .86.46l-1.92 6.02A1 1 0 0 0 13 10h7a1 1 0 0 1 .78 1.63l-9.9 10.2a.5.5 0 0 1-.86-.46l1.92-6.02A1 1 0 0 0 11 14z"></path></svg>
--------------------------------------------------------------------------------
[A-9]
   🔗 href       : https://dpm.lol/champions/Braum/build/pro
   🎨 class      : 
   🧭 xpath      : //*[@id="root"]/main[1]/div[1]/div[1]/div[1]/div[1]/div[1]/a[2]
   🏷️ span text : None
   🎨 SVG #1:
<svg xmlns="http://www.w3.org/2000/svg" width

In [53]:
from selenium.webdriver.common.by import By
import hashlib


def get_selected_played_lane(driver) -> dict:
    print("🟢 Détection de la played lane sélectionnée")

    # 1️⃣ on récupère TOUS les containers possibles (sécurise si la page évolue)
    containers = driver.find_elements(
        By.CSS_SELECTOR,
        "div.border-black-600.flex.w-fit.items-center.justify-center"
    )

    print(f"🔍 {len(containers)} containers candidats trouvés")

    if not containers:
        print("❌ Aucun container trouvé")
        return {"lane": None, "percentage": None}

    # 2️⃣ on prend le premier (structure unique sur la page)
    container = containers[0]

    lane_divs = container.find_elements(By.XPATH, "./div")
    print(f"🔍 {len(lane_divs)} boutons de lane trouvés")

    # 3️⃣ on parcourt uniquement les 5 boutons
    for idx, lane_div in enumerate(lane_divs):
        try:
            link = lane_div.find_element(By.TAG_NAME, "a")
            inner_div = link.find_element(By.TAG_NAME, "div")

            class_name = inner_div.get_attribute("class") or ""
            print(f"[Lane {idx}] classes = {class_name}")

            # pas sélectionné → on skip
            if "bg-blue-200" not in class_name:
                continue

            print(f"⭐ Lane sélectionnée détectée (index {idx})")

            # SVG → hash → lane
            svg = inner_div.find_element(By.TAG_NAME, "svg")
            svg_hash = hash_svg_path(svg)
            lane = resolve_role_from_hash(svg_hash)

            print(f"   🔐 svg_hash = {svg_hash}")
            print(f"   🏷️ lane     = {lane}")

            # 4️⃣ span JUSTE APRÈS le <a>
            percentage = None
            try:
                span = lane_div.find_element(By.TAG_NAME, "span")
                percentage = span.text.strip()
            except Exception:
                print("⚠️ Span pourcentage introuvable")

            print(f"   📊 percentage = {percentage}")

            return {
                "lane": lane,
                "percentage": percentage
            }

        except Exception as e:
            print(f"[Lane {idx}] ❌ erreur : {e}")

    print("❌ Aucune lane sélectionnée trouvée")
    return {"lane": None, "percentage": None}


In [54]:
get_selected_played_lane(driver)

🟢 Détection de la played lane sélectionnée
🔍 2 containers candidats trouvés
🔍 5 boutons de lane trouvés
[Lane 0] classes = p-8
[Lane 1] classes = p-8 opacity-50 cursor-default
[Lane 2] classes = p-8 opacity-50 cursor-default
[Lane 3] classes = p-8 opacity-50 cursor-default
[Lane 4] classes = p-8 text-black-900 rounded-md bg-blue-200/90
⭐ Lane sélectionnée détectée (index 4)
   🔐 svg_hash = 3b66426a9b2beeca218cc726985f68b1
   🏷️ lane     = sup
   📊 percentage = 99.9%


{'lane': 'sup', 'percentage': '99.9%'}

In [26]:
def hash_svg_path(svg_element) -> str:
    """
    Extrait le path 'd' du SVG et retourne son hash MD5
    """
    paths = svg_element.find_elements(By.TAG_NAME, "path")
    if not paths:
        return None

    path_d = paths[0].get_attribute("d").strip()
    return hashlib.md5(path_d.encode("utf-8")).hexdigest()


In [42]:
#ancien

# ROLE_HASH_TO_TEXT = {
#     "54d7bacd7686d25f9555c3381d5b3ecb": "jungle",
#     "d1a365179625b6191d515c69f5277dbd": "support",
#     "f84094b0fe98e4bf44fe62648e255e41": "adc",
#     "6f7f06ca1bef87e71a35726cf843ad87": "top",
#     "e4f796e42865301ea9dd362f979a2cdc": "mid",
# }

#nouveau en considérant le path du svg et pas son hash global : 
ROLE_HASH_TO_TEXT = {
    "df14d23b35c9842bd8afa0db2b922aaf": "top",
    "bd9e54c883010f9bc2487e7d26a91b77": "jun",
    "3ce111209b6d69bee8498e94b02567ad": "mid",
    "6f1d8859e29002c2c45b527544e5a755": "adc",
    "3b66426a9b2beeca218cc726985f68b1": "sup",
}

def resolve_role_from_hash(svg_hash: str) -> str:
    role = ROLE_HASH_TO_TEXT.get(svg_hash)

    if role is None:
        print(f"⚠️ Hash de rôle inconnu : {svg_hash}")
        return "unknown"

    return role

# svg_hash = hashlib.md5(role_svg.encode("utf-8")).hexdigest()


#nouveau (avec le path du svg)

def hash_svg_path(svg_element) -> str:
    """
    Extrait le path 'd' du SVG et retourne son hash MD5
    """
    paths = svg_element.find_elements(By.TAG_NAME, "path")
    if not paths:
        return None

    path_d = paths[0].get_attribute("d").strip()
    return hashlib.md5(path_d.encode("utf-8")).hexdigest()


In [39]:
def click_matchup_or_synergy_lane(driver, lane_name) -> bool:
    print(f"🟢 Sélection de la lane {lane_name} via hash du PATH SVG")

    time.sleep(0.5)

    lane_container_xpath = (
        '//*[@id="root"]/main[1]/div[1]/div[3]/div[2]/div[1]/div[2]/div[4]'
        '/div[1]/div[1]/div[3]/div[1]'
    )

    try:
        lane_container = driver.find_element(By.XPATH, lane_container_xpath)
    except Exception as e:
        print(f"❌ Conteneur des lanes introuvable : {e}")
        return False

    lane_buttons = lane_container.find_elements(By.TAG_NAME, "button")
    print(f"🔍 {len(lane_buttons)} boutons de lane trouvés\n")

    for i, btn in enumerate(lane_buttons):
        svgs = btn.find_elements(By.TAG_NAME, "svg")
        if not svgs:
            continue

        svg = svgs[0]
        svg_hash = hash_svg_path(svg)

        if not svg_hash:
            continue

        role = resolve_role_from_hash(svg_hash)

        print(f"[Lane {i}] hash={svg_hash} role={role}")

        if role == lane_name:
            print(f"🎯 Bouton {lane_name} identifié — clic")

            driver.execute_script(
                "arguments[0].scrollIntoView({block: 'center'});",
                btn,
            )
            time.sleep(0.3)
            btn.click()

            print(f"🖱️ Bouton {lane_name} cliqué avec succès")
            return True

    print(f"❌ Bouton {lane_name} non trouvé")
    return False


In [43]:
click_matchup_or_synergy_lane(driver, "sup")

🟢 Sélection de la lane sup via hash du PATH SVG
🔍 5 boutons de lane trouvés

[Lane 0] hash=df14d23b35c9842bd8afa0db2b922aaf role=top
[Lane 1] hash=bd9e54c883010f9bc2487e7d26a91b77 role=jun
[Lane 2] hash=3ce111209b6d69bee8498e94b02567ad role=mid
[Lane 3] hash=6f1d8859e29002c2c45b527544e5a755 role=adc
[Lane 4] hash=3b66426a9b2beeca218cc726985f68b1 role=sup
🎯 Bouton sup identifié — clic
🖱️ Bouton sup cliqué avec succès


True

---
test db Aissam

#### D:\lol draft analyzer - datascientest\db_Aïssam_0402\lol_matches.db
---

In [5]:
import sqlite3

db_path = r"D:\lol draft analyzer - datascientest\db_Aïssam_0402\lol_matches.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print("Connexion OK")


OperationalError: unable to open database file

In [6]:

import sqlite3
import pandas as pd

# Connexion
db_path = r"D:\lol draft analyzer - datascientest\db_Aïssam_0402\lol_matches.db"
conn = sqlite3.connect(db_path)

# Lire une table entière
df = pd.read_sql("SELECT * FROM matches LIMIT 1000", conn)

# Fermer la connexion
conn.close()

OperationalError: unable to open database file

In [11]:
import sqlite3
import pandas as pd
import os
import sys

print("=== DEBUG SQLITE ===")

db_path = r"..\..\data\lol_matches.db"

print("1️⃣ db_path :", db_path)
print("2️⃣ cwd (répertoire courant) :", os.getcwd())
print("3️⃣ chemin absolu :", os.path.abspath(db_path))
print("4️⃣ exists :", os.path.exists(db_path))
print("5️⃣ isfile :", os.path.isfile(db_path))

# Lister le contenu du dossier data
data_dir = os.path.dirname(os.path.abspath(db_path))
print("6️⃣ contenu du dossier data :")
if os.path.exists(data_dir):
    print(os.listdir(data_dir))
else:
    print("❌ dossier data introuvable")

print("7️⃣ tentative connexion sqlite...")

try:
    conn = sqlite3.connect(
        f"file:{os.path.abspath(db_path)}?mode=ro",
        uri=True
    )
    print("✅ Connexion réussie")

    print("8️⃣ test requête tables")
    tables = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table';",
        conn
    )
    print(tables)

    print("9️⃣ test lecture table matches")
    df = pd.read_sql(
        "SELECT * FROM matches LIMIT 5;",
        conn
    )
    print(df.head())

    conn.close()
    print("🔚 Connexion fermée")

except Exception as e:
    print("❌ ERREUR :", type(e).__name__)
    print(e)


=== DEBUG SQLITE ===
1️⃣ db_path : ..\..\data\lol_matches.db
2️⃣ cwd (répertoire courant) : c:\Users\samue\Documents\datascientest-lol-draft_analyzer\src\notebooks
3️⃣ chemin absolu : c:\Users\samue\Documents\datascientest-lol-draft_analyzer\data\lol_matches.db
4️⃣ exists : True
5️⃣ isfile : True
6️⃣ contenu du dossier data :
['bases_annexes', 'champion_metadata.json', 'Dataframes', 'lol_matches-old.db', 'lol_matches.db', 'old']
7️⃣ tentative connexion sqlite...
✅ Connexion réussie
8️⃣ test requête tables
                    name
0                matches
1             team_stats
2        sqlite_sequence
3           player_stats
4    collection_progress
5       collection_stats
6              summoners
7   summoner_elo_history
8       champion_mastery
9                patches
10  champion_patch_stats
11        match_timeline
12     champion_matchups
13    champion_synergies
14          match_events
9️⃣ test lecture table matches
          match_id  game_creation  game_duration    game_v